In [ ]:
# 3D HDNet

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
#from torchvision import datasets, transforms
import os
import numpy as np
import sys
import json
import astra
import time
import matplotlib.pyplot as plt
from pathlib import Path
import skimage


class double_conv3d_bn(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,out_channels,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(out_channels,out_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class double_conv3d_bn_bot(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn_bot,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,out_channels,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(out_channels,in_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(in_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class double_conv3d_bn_red_block(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn_red_block,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,in_channels//2,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(in_channels//2,out_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(in_channels//2, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class deconv3d_bn(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=2):

        super(deconv3d_bn,self).__init__()

        self.conv1 = nn.ConvTranspose3d(in_channels, out_channels, kernel_size = kernel_size, stride = strides, 
                                        padding = 1, output_padding = 1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class multi_scale(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(multi_scale, self).__init__()

        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)

        self.convx = nn.ConvTranspose3d(in_channels=1, out_channels=1, kernel_size=(1, 2, 1), stride=(1, 2, 1), padding=(0, 0, 0))
        self.bnx = nn.BatchNorm3d(out_channels, track_running_stats = False)     
        
        self.upsample = nn.Upsample(size=(32, 64, 32), mode='trilinear', align_corners=True)

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        
        '''
        x = self.convx(x)
        x = self.bnx(x)
        x = F.leaky_relu(x, 0.2)
        '''
        x = self.upsample(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)

        return x


class HDNet(nn.Module):
    def __init__(self):

        super(HDNet,self).__init__()

        self.layer1_conv = double_conv3d_bn(1, 32)
        self.layer2_conv = double_conv3d_bn(32, 64)
        self.layer3_conv = double_conv3d_bn(64, 128)
        self.layer4_conv = double_conv3d_bn(128, 256)

        self.layer5_conv = double_conv3d_bn_bot(256, 512)

        self.layer6_conv = double_conv3d_bn_red_block(512, 128)
        self.layer7_conv = double_conv3d_bn_red_block(256, 64)
        self.layer8_conv = double_conv3d_bn_red_block(128, 32)
        self.layer9_conv = double_conv3d_bn_red_block(64, 32)

        self.layer10_conv = nn.Conv3d(32, 2, kernel_size=3, stride=1, padding='same', bias=False)

        self.c1 = nn.Conv3d(32, 32, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c2 = nn.Conv3d(64, 64, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c3 = nn.Conv3d(128, 128, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c4 = nn.Conv3d(256, 256, kernel_size=3, stride = 2, padding = 1, bias=False)

        self.deconv1 = deconv3d_bn(256, 256)
        self.deconv2 = deconv3d_bn(128, 128)
        self.deconv3 = deconv3d_bn(64, 64)
        self.deconv4 = deconv3d_bn(32, 32)
        
        self.clast = nn.Conv3d(2, 1, kernel_size=3, stride = 1, padding = 'same', bias=False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()
        
        self.multi_scale_layer = multi_scale(1, 1)

        #self.fc = nn.Linear(8192, 10) # 10 classes

        #self.sigmoid = nn.Sigmoid()

    def forward(self,x):
        
        #print('input', x.size())
        
        inp_x = x

        conv1 = self.layer1_conv(x)
        pool1 = self.c1(conv1)
        
        #print('conv1', conv1.size())

        conv2 = self.layer2_conv(pool1)
        pool2 = self.c2(conv2)
        
        #print('conv2', conv2.size())

        conv3 = self.layer3_conv(pool2)
        pool3 = self.c3(conv3)
        
        #print('conv3', conv3.size())

        conv4 = self.layer4_conv(pool3)
        pool4 = self.c4(conv4)
        
        #print('conv4', conv4.size())

        conv5 = self.layer5_conv(pool4)   
        
        #print('conv5', conv5.size())

        #print('conv5', conv5.size())
        # in  =  2
        # out = (in - 1) * s - 2 * p + d * (k - 1) + out_p + 1 
        #        2         1       0   1    3        0
        #     =  4
        convt1 = self.deconv1(conv5)    
        
        #print('convt1', convt1.size())
        #print('conv4', conv4.size())
        
        concat1 = torch.cat([convt1, conv4[:,:,:,:]], dim=1)
        conv6 = self.layer6_conv(concat1)

        #print('conv6', conv6.size())
        # in  =  4
        # out = (in - 1) * s - 2 * p + d * (k - 1) + out_p + 1 
        #        4         1       0   1    3        0
        #     =  6
        convt2 = self.deconv2(conv6)
        
        #print('convt2', convt2.size())
        #print('conv3', conv3.size())
        
        concat2 = torch.cat([convt2, conv3[:,:,:,:]], dim=1)
        conv7 = self.layer7_conv(concat2)

        convt3 = self.deconv3(conv7)
        concat3 = torch.cat([convt3, conv2[:,:,:,:]], dim=1)
        conv8 = self.layer8_conv(concat3)

        convt4 = self.deconv4(conv8)
        concat4 = torch.cat([convt4, conv1[:,:,:,:]], dim=1)
        conv9 = self.layer9_conv(concat4)
        outp = self.layer10_conv(conv9)
        
        outp = self.clast(outp)
        
        #outp = torch.flatten(outp, 1)
        #print('flatten:' , outp.size())
        #outp = self.fc(outp)
        #print('fc:' , outp.size())
        #outp = F.log_softmax(outp, dim=1) # log prob for numerical stability
        
        outp = F.leaky_relu(outp, 0.2)
        #outp = self.tanh(outp) # replace leaky_relu to tanh
        #outp = self.elu(outp) # replace leaky_relu to elu
        
        # RESIDUAL
        #outp = outp + inp_x
        #
        
        #outp = self.multi_scale_layer(outp)
        
        return outp
    
model = HDNet()
inp = torch.rand(1, 1, 32, 32, 32)
print('input: ', inp.shape)
print('output: ', model(inp).shape)

In [ ]:
# 3D HDNet+ (residual)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
#from torchvision import datasets, transforms
import os
import numpy as np
import sys
import json
import astra
import time
import matplotlib.pyplot as plt
from pathlib import Path
import skimage


class double_conv3d_bn(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,out_channels,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(out_channels,out_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class double_conv3d_bn_bot(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn_bot,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,out_channels,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(out_channels,in_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(in_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class double_conv3d_bn_red_block(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn_red_block,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,in_channels//2,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(in_channels//2,out_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(in_channels//2, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class deconv3d_bn(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=2):

        super(deconv3d_bn,self).__init__()

        self.conv1 = nn.ConvTranspose3d(in_channels, out_channels, kernel_size = kernel_size, stride = strides, 
                                        padding = 1, output_padding = 1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class multi_scale(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(multi_scale, self).__init__()

        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)

        self.convx = nn.ConvTranspose3d(in_channels=1, out_channels=1, kernel_size=(1, 2, 1), stride=(1, 2, 1), padding=(0, 0, 0))
        self.bnx = nn.BatchNorm3d(out_channels, track_running_stats = False)     
        
        self.upsample = nn.Upsample(size=(32, 64, 32), mode='trilinear', align_corners=True)

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        
        '''
        x = self.convx(x)
        x = self.bnx(x)
        x = F.leaky_relu(x, 0.2)
        '''
        x = self.upsample(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)

        return x


class HDNet(nn.Module):
    def __init__(self):

        super(HDNet,self).__init__()

        self.layer1_conv = double_conv3d_bn(1, 32)
        self.layer2_conv = double_conv3d_bn(32, 64)
        self.layer3_conv = double_conv3d_bn(64, 128)
        self.layer4_conv = double_conv3d_bn(128, 256)

        self.layer5_conv = double_conv3d_bn_bot(256, 512)

        self.layer6_conv = double_conv3d_bn_red_block(512, 128)
        self.layer7_conv = double_conv3d_bn_red_block(256, 64)
        self.layer8_conv = double_conv3d_bn_red_block(128, 32)
        self.layer9_conv = double_conv3d_bn_red_block(64, 32)

        self.layer10_conv = nn.Conv3d(32, 2, kernel_size=3, stride=1, padding='same', bias=False)

        self.c1 = nn.Conv3d(32, 32, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c2 = nn.Conv3d(64, 64, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c3 = nn.Conv3d(128, 128, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c4 = nn.Conv3d(256, 256, kernel_size=3, stride = 2, padding = 1, bias=False)

        self.deconv1 = deconv3d_bn(256, 256)
        self.deconv2 = deconv3d_bn(128, 128)
        self.deconv3 = deconv3d_bn(64, 64)
        self.deconv4 = deconv3d_bn(32, 32)
        
        self.clast = nn.Conv3d(2, 1, kernel_size=3, stride = 1, padding = 'same', bias=False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()
        
        self.multi_scale_layer = multi_scale(1, 1)

        #self.fc = nn.Linear(8192, 10) # 10 classes

        #self.sigmoid = nn.Sigmoid()

    def forward(self,x):
        
        #print('input', x.size())
        
        inp_x = x

        conv1 = self.layer1_conv(x)
        pool1 = self.c1(conv1)
        
        #print('conv1', conv1.size())

        conv2 = self.layer2_conv(pool1)
        pool2 = self.c2(conv2)
        
        #print('conv2', conv2.size())

        conv3 = self.layer3_conv(pool2)
        pool3 = self.c3(conv3)
        
        #print('conv3', conv3.size())

        conv4 = self.layer4_conv(pool3)
        pool4 = self.c4(conv4)
        
        #print('conv4', conv4.size())

        conv5 = self.layer5_conv(pool4)   
        
        #print('conv5', conv5.size())

        #print('conv5', conv5.size())
        # in  =  2
        # out = (in - 1) * s - 2 * p + d * (k - 1) + out_p + 1 
        #        2         1       0   1    3        0
        #     =  4
        convt1 = self.deconv1(conv5)    
        
        #print('convt1', convt1.size())
        #print('conv4', conv4.size())
        
        concat1 = torch.cat([convt1, conv4[:,:,:,:]], dim=1)
        conv6 = self.layer6_conv(concat1)

        #print('conv6', conv6.size())
        # in  =  4
        # out = (in - 1) * s - 2 * p + d * (k - 1) + out_p + 1 
        #        4         1       0   1    3        0
        #     =  6
        convt2 = self.deconv2(conv6)
        
        #print('convt2', convt2.size())
        #print('conv3', conv3.size())
        
        concat2 = torch.cat([convt2, conv3[:,:,:,:]], dim=1)
        conv7 = self.layer7_conv(concat2)

        convt3 = self.deconv3(conv7)
        concat3 = torch.cat([convt3, conv2[:,:,:,:]], dim=1)
        conv8 = self.layer8_conv(concat3)

        convt4 = self.deconv4(conv8)
        concat4 = torch.cat([convt4, conv1[:,:,:,:]], dim=1)
        conv9 = self.layer9_conv(concat4)
        outp = self.layer10_conv(conv9)
        
        outp = self.clast(outp)
        
        #outp = torch.flatten(outp, 1)
        #print('flatten:' , outp.size())
        #outp = self.fc(outp)
        #print('fc:' , outp.size())
        #outp = F.log_softmax(outp, dim=1) # log prob for numerical stability
        
        outp = F.leaky_relu(outp, 0.2)
        #outp = self.tanh(outp) # replace leaky_relu to tanh
        #outp = self.elu(outp) # replace leaky_relu to elu
        
        # RESIDUAL
        outp = outp + inp_x
        #
        
        #outp = self.multi_scale_layer(outp)
        
        return outp
    
model = HDNet()
inp = torch.rand(1, 1, 32, 32, 32)
print('input: ', inp.shape)
print('output: ', model(inp).shape)

In [ ]:
# 3D MSNet (projection domain)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
#from torchvision import datasets, transforms
import os
import numpy as np
import sys
import json
import astra
import time
import matplotlib.pyplot as plt
from pathlib import Path
import skimage
from PIL import Image


new_size = (32, 64, 64)

class double_conv3d_bn(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,out_channels,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(out_channels,out_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class double_conv3d_bn_bot(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn_bot,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,out_channels,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(out_channels,in_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(in_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class double_conv3d_bn_red_block(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn_red_block,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,in_channels//2,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(in_channels//2,out_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(in_channels//2, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class deconv3d_bn(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=2):

        super(deconv3d_bn,self).__init__()

        self.conv1 = nn.ConvTranspose3d(in_channels, out_channels, kernel_size = kernel_size, stride = strides, 
                                        padding = 1, output_padding = 1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x
'''
class multi_scale(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=2):

        super(multi_scale, self).__init__()

        # 32,32,32 => 32,64,64
        #self.conv1 = nn.ConvTranspose3d(1, 1, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=1, output_padding=(0, 1, 1), bias=False)

        # 32,32,32 => 32,64,32
        self.conv1 = nn.ConvTranspose3d(in_channels=1, out_channels=1, kernel_size=(1, 2, 1), stride=(1, 2, 1), padding=(0, 0, 0))

        self.bn1 = nn.BatchNorm3d(out_channels)

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        #x = F.leaky_relu(x, 0.2)

        return x
'''

class multi_scale(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(multi_scale, self).__init__()

        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)

        self.convx = nn.ConvTranspose3d(in_channels=1, out_channels=1, kernel_size=(1, 2, 1), stride=(1, 2, 1), padding=(0, 0, 0))
        self.bnx = nn.BatchNorm3d(out_channels, track_running_stats = False)     
        
        self.upsample = nn.Upsample(size=(32, 64, 32), mode='trilinear', align_corners=True)

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        
        '''
        x = self.convx(x)
        x = self.bnx(x)
        x = F.leaky_relu(x, 0.2)
        '''
        x = self.upsample(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)

        return x
    
    
class MSNet_multi(nn.Module):
    def __init__(self):

        super(MSNet_multi,self).__init__()

        self.layer1_conv = double_conv3d_bn(1, 32)
        self.layer2_conv = double_conv3d_bn(32, 64)
        self.layer3_conv = double_conv3d_bn(64, 128)
        self.layer4_conv = double_conv3d_bn(128, 256)

        self.layer5_conv = double_conv3d_bn_bot(256, 512)

        self.layer6_conv = double_conv3d_bn_red_block(512, 128)
        self.layer7_conv = double_conv3d_bn_red_block(256, 64)
        self.layer8_conv = double_conv3d_bn_red_block(128, 32)
        self.layer9_conv = double_conv3d_bn_red_block(64, 32)

        self.layer10_conv = nn.Conv3d(32, 2, kernel_size=3, stride=1, padding='same', bias=False)

        self.c1 = nn.Conv3d(32, 32, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c2 = nn.Conv3d(64, 64, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c3 = nn.Conv3d(128, 128, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c4 = nn.Conv3d(256, 256, kernel_size=3, stride = 2, padding = 1, bias=False)

        self.deconv1 = deconv3d_bn(256, 256)
        self.deconv2 = deconv3d_bn(128, 128)
        self.deconv3 = deconv3d_bn(64, 64)
        self.deconv4 = deconv3d_bn(32, 32)
        
        self.clast = nn.Conv3d(2, 1, kernel_size=3, stride = 1, padding = 'same', bias=False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()
        
        self.multi_scale_layer = multi_scale(1, 1)

        #self.fc = nn.Linear(8192, 10) # 10 classes

        #self.sigmoid = nn.Sigmoid()

    def forward(self,x):
        
        #print('input', x.size())
        
        inp_x = x
        inp_x = F.interpolate(inp_x, size=(32, 64, 32), mode='trilinear', align_corners=True)

        x = self.multi_scale_layer(x)
        x += inp_x

        conv1 = self.layer1_conv(x)
        pool1 = self.c1(conv1)
        
        #print('conv1', conv1.size())

        conv2 = self.layer2_conv(pool1)
        pool2 = self.c2(conv2)
        
        #print('conv2', conv2.size())

        conv3 = self.layer3_conv(pool2)
        pool3 = self.c3(conv3)
        
        #print('conv3', conv3.size())

        conv4 = self.layer4_conv(pool3)
        pool4 = self.c4(conv4)
        
        #print('conv4', conv4.size())

        conv5 = self.layer5_conv(pool4)   
        
        #print('conv5', conv5.size())

        #print('conv5', conv5.size())
        # in  =  2
        # out = (in - 1) * s - 2 * p + d * (k - 1) + out_p + 1 
        #        2         1       0   1    3        0
        #     =  4
        convt1 = self.deconv1(conv5)    
        
        #print('convt1', convt1.size())
        #print('conv4', conv4.size())
        
        concat1 = torch.cat([convt1, conv4[:,:,:,:]], dim=1)
        conv6 = self.layer6_conv(concat1)

        #print('conv6', conv6.size())
        # in  =  4
        # out = (in - 1) * s - 2 * p + d * (k - 1) + out_p + 1 
        #        4         1       0   1    3        0
        #     =  6
        convt2 = self.deconv2(conv6)
        
        #print('convt2', convt2.size())
        #print('conv3', conv3.size())
        
        concat2 = torch.cat([convt2, conv3[:,:,:,:]], dim=1)
        conv7 = self.layer7_conv(concat2)

        convt3 = self.deconv3(conv7)
        concat3 = torch.cat([convt3, conv2[:,:,:,:]], dim=1)
        conv8 = self.layer8_conv(concat3)

        convt4 = self.deconv4(conv8)
        concat4 = torch.cat([convt4, conv1[:,:,:,:]], dim=1)
        conv9 = self.layer9_conv(concat4)
        outp = self.layer10_conv(conv9)
        
        outp = self.clast(outp)
        
        #outp = torch.flatten(outp, 1)
        #print('flatten:' , outp.size())
        #outp = self.fc(outp)
        #print('fc:' , outp.size())
        #outp = F.log_softmax(outp, dim=1) # log prob for numerical stability
        
        outp = F.leaky_relu(outp, 0.2)
        #outp = self.tanh(outp) # replace leaky_relu to tanh
        #outp = self.elu(outp) # replace leaky_relu to elu
        
        # RESIDUAL
        #outp = outp + inp_x
        #
        
        
        #inp_x = F.interpolate(inp_x.squeeze(0).squeeze(0), size=new_size, mode='linear', align_corners=True)
        #inp_x = inp_x.squeeze(0).squeeze(0)
        #inp_x = inp_x.permute(0, 2, 1)
        #inp_x = F.interpolate(inp_x, size=(32,64,32), mode='trilinear', align_corners=True) 
        #inp_x = F.interpolate(inp_x, size=(64,128,64), mode='trilinear', align_corners=True)
        #inp_x = F.interpolate(inp_x, size=64, mode='linear', align_corners=True)
        #inp_x = inp_x.permute(0, 2, 1)
        #inp_x = inp_x.unsqueeze(0).unsqueeze(0)
        
        
        #outp = self.multi_scale_layer(outp)
        #inp_x = self.multi_scale_layer(inp_x)
        #outp = outp + inp_x
        #outp = F.leaky_relu(outp, 0.2)
        
        #################
        outp = outp + inp_x
        #################
        
        return outp

model = MSNet_multi()
inp = torch.rand(1, 1, 32, 32, 32)
print('input: ', inp.shape)
print('output: ', model(inp).shape)


In [ ]:
# 3D MSNet (image domain)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
#from torchvision import datasets, transforms
import os
import numpy as np
import sys
import json
import astra
import time
import matplotlib.pyplot as plt
from pathlib import Path
import skimage
from PIL import Image


new_size = (32, 64, 64)

class double_conv3d_bn(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,out_channels,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(out_channels,out_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class double_conv3d_bn_bot(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn_bot,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,out_channels,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(out_channels,in_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(in_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class double_conv3d_bn_red_block(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=1,padding=1):

        super(double_conv3d_bn_red_block,self).__init__()

        self.conv1 = nn.Conv3d(in_channels,in_channels//2,
                              kernel_size=kernel_size,
                              stride = strides,padding='same',bias=False)
        self.conv2 = nn.Conv3d(in_channels//2,out_channels,
                              kernel_size = kernel_size,
                              stride = strides,padding='same',bias=False)
        self.bn1 = nn.BatchNorm3d(in_channels//2, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu
        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x


class deconv3d_bn(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=2):

        super(deconv3d_bn,self).__init__()

        self.conv1 = nn.ConvTranspose3d(in_channels, out_channels, kernel_size = kernel_size, stride = strides, 
                                        padding = 1, output_padding = 1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        #x = self.tanh(x) # replace leaky_relu to tanh
        #x = self.elu(x) # replace leaky_relu to elu

        return x
'''
class multi_scale(nn.Module):
    def __init__(self,in_channels,out_channels,kernel_size=3,strides=2):

        super(multi_scale, self).__init__()

        # 32,32,32 => 32,64,64
        #self.conv1 = nn.ConvTranspose3d(1, 1, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=1, output_padding=(0, 1, 1), bias=False)

        # 32,32,32 => 32,64,32
        self.conv1 = nn.ConvTranspose3d(in_channels=1, out_channels=1, kernel_size=(1, 2, 1), stride=(1, 2, 1), padding=(0, 0, 0))

        self.bn1 = nn.BatchNorm3d(out_channels)

    def forward(self,x):

        x = self.conv1(x)
        x = self.bn1(x)
        #x = F.leaky_relu(x, 0.2)

        return x
'''

class multi_scale(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(multi_scale, self).__init__()

        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm3d(out_channels, track_running_stats = False)
        self.bn2 = nn.BatchNorm3d(out_channels, track_running_stats = False)

        self.convx = nn.ConvTranspose3d(in_channels=1, out_channels=1, kernel_size=(1, 2, 1), stride=(1, 2, 1), padding=(0, 0, 0))
        self.bnx = nn.BatchNorm3d(out_channels, track_running_stats = False)     
        
        self.upsample = nn.Upsample(size=(32, 64, 32), mode='trilinear', align_corners=True)

    def forward(self, x):

        x = self.conv1(x)
        x = self.bn1(x)
        x = F.leaky_relu(x, 0.2)
        
        '''
        x = self.convx(x)
        x = self.bnx(x)
        x = F.leaky_relu(x, 0.2)
        '''
        x = self.upsample(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = F.leaky_relu(x, 0.2)

        return x
    
    
class MSNet_multi(nn.Module):
    def __init__(self):

        super(MSNet_multi,self).__init__()

        self.layer1_conv = double_conv3d_bn(1, 32)
        self.layer2_conv = double_conv3d_bn(32, 64)
        self.layer3_conv = double_conv3d_bn(64, 128)
        self.layer4_conv = double_conv3d_bn(128, 256)

        self.layer5_conv = double_conv3d_bn_bot(256, 512)

        self.layer6_conv = double_conv3d_bn_red_block(512, 128)
        self.layer7_conv = double_conv3d_bn_red_block(256, 64)
        self.layer8_conv = double_conv3d_bn_red_block(128, 32)
        self.layer9_conv = double_conv3d_bn_red_block(64, 32)

        self.layer10_conv = nn.Conv3d(32, 2, kernel_size=3, stride=1, padding='same', bias=False)

        self.c1 = nn.Conv3d(32, 32, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c2 = nn.Conv3d(64, 64, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c3 = nn.Conv3d(128, 128, kernel_size=3, stride = 2, padding = 1, bias=False)
        self.c4 = nn.Conv3d(256, 256, kernel_size=3, stride = 2, padding = 1, bias=False)

        self.deconv1 = deconv3d_bn(256, 256)
        self.deconv2 = deconv3d_bn(128, 128)
        self.deconv3 = deconv3d_bn(64, 64)
        self.deconv4 = deconv3d_bn(32, 32)
        
        self.clast = nn.Conv3d(2, 1, kernel_size=3, stride = 1, padding = 'same', bias=False)
        
        self.tanh = nn.Tanh()
        self.elu = nn.ELU()
        
        self.multi_scale_layer = multi_scale(1, 1)

        #self.fc = nn.Linear(8192, 10) # 10 classes

        #self.sigmoid = nn.Sigmoid()

    def forward(self,x):
        
        #print('input', x.size())
        
        inp_x = x
        #inp_x = F.interpolate(inp_x, size=(32, 64, 32), mode='trilinear', align_corners=True)

        #x = self.multi_scale_layer(x)
        #x += inp_x

        conv1 = self.layer1_conv(x)
        pool1 = self.c1(conv1)
        
        #print('conv1', conv1.size())

        conv2 = self.layer2_conv(pool1)
        pool2 = self.c2(conv2)
        
        #print('conv2', conv2.size())

        conv3 = self.layer3_conv(pool2)
        pool3 = self.c3(conv3)
        
        #print('conv3', conv3.size())

        conv4 = self.layer4_conv(pool3)
        pool4 = self.c4(conv4)
        
        #print('conv4', conv4.size())

        conv5 = self.layer5_conv(pool4)   
        
        #print('conv5', conv5.size())

        #print('conv5', conv5.size())
        # in  =  2
        # out = (in - 1) * s - 2 * p + d * (k - 1) + out_p + 1 
        #        2         1       0   1    3        0
        #     =  4
        convt1 = self.deconv1(conv5)    
        
        #print('convt1', convt1.size())
        #print('conv4', conv4.size())
        
        concat1 = torch.cat([convt1, conv4[:,:,:,:]], dim=1)
        conv6 = self.layer6_conv(concat1)

        #print('conv6', conv6.size())
        # in  =  4
        # out = (in - 1) * s - 2 * p + d * (k - 1) + out_p + 1 
        #        4         1       0   1    3        0
        #     =  6
        convt2 = self.deconv2(conv6)
        
        #print('convt2', convt2.size())
        #print('conv3', conv3.size())
        
        concat2 = torch.cat([convt2, conv3[:,:,:,:]], dim=1)
        conv7 = self.layer7_conv(concat2)

        convt3 = self.deconv3(conv7)
        concat3 = torch.cat([convt3, conv2[:,:,:,:]], dim=1)
        conv8 = self.layer8_conv(concat3)

        convt4 = self.deconv4(conv8)
        concat4 = torch.cat([convt4, conv1[:,:,:,:]], dim=1)
        conv9 = self.layer9_conv(concat4)
        outp = self.layer10_conv(conv9)
        
        outp = self.clast(outp)
        
        #outp = torch.flatten(outp, 1)
        #print('flatten:' , outp.size())
        #outp = self.fc(outp)
        #print('fc:' , outp.size())
        #outp = F.log_softmax(outp, dim=1) # log prob for numerical stability
        
        outp = F.leaky_relu(outp, 0.2)
        #outp = self.tanh(outp) # replace leaky_relu to tanh
        #outp = self.elu(outp) # replace leaky_relu to elu
        
        # RESIDUAL
        #outp = outp + inp_x
        #
        
        
        #inp_x = F.interpolate(inp_x.squeeze(0).squeeze(0), size=new_size, mode='linear', align_corners=True)
        #inp_x = inp_x.squeeze(0).squeeze(0)
        #inp_x = inp_x.permute(0, 2, 1)
        #inp_x = F.interpolate(inp_x, size=(32,64,32), mode='trilinear', align_corners=True) 
        #inp_x = F.interpolate(inp_x, size=(64,128,64), mode='trilinear', align_corners=True)
        #inp_x = F.interpolate(inp_x, size=64, mode='linear', align_corners=True)
        #inp_x = inp_x.permute(0, 2, 1)
        #inp_x = inp_x.unsqueeze(0).unsqueeze(0)
        
        
        #outp = self.multi_scale_layer(outp)
        #inp_x = self.multi_scale_layer(inp_x)
        #outp = outp + inp_x
        #outp = F.leaky_relu(outp, 0.2)
        
        #################
        outp = outp + inp_x
        #################
        
        return outp

model = MSNet_multi()
inp = torch.rand(1, 1, 32, 32, 32)
print('input: ', inp.shape)
print('output: ', model(inp).shape)


In [ ]:
# Cone.py 

# cone-beam tomography 

# (c) 2022, Chang-Chieh Cheng, jameschengcs@nycu.edu.tw



import numpy as np

import copy

import sys

sys.path.append('../')

#import vgi

import astra



__all__ = ('ConeRec', 'astraProjShape', )

 

 # From (views, slices, detectors) = (slices, views, detectors)

def astraProjShape(proj):

    return np.swapaxes(proj, 0, 1)    



# ----------------------------------------------------

# Projection shapes: (slices, angles, detectors)

class ConeRec:

    def __init__(self, vol_shape, proj_shape, scan_range = (0, 2 * np.pi), angles = None, volume = None, proj = None,

                 det_width = 1.0, source_origin = 512., origin_det = 512.,

                 algo = 'SIRT3D_CUDA', iterations = 1000):

        self.vol_shape = vol_shape      # (d, h, w)

        self.depth, self.height, self.width = self.vol_shape

        self.proj_shape = proj_shape    # (slices, angles, detectors)

        self.n_det_rows, self.n_angles, self.n_det_cols = self.proj_shape

        self.scan_range = scan_range

        self.proj_mode = 'cone'      

        # create_vol_geom(Y, X, Z)``:  

        self.vol_geom = astra.create_vol_geom(self.height, self.width, self.depth)

        self.vol_id = astra.data3d.create('-vol', self.vol_geom, data = volume)

        if angles is None:

            self.angles = np.linspace(self.scan_range[0], self.scan_range[1], self.n_angles, False)

        else:

            self.angles = angles

        self.det_width = det_width

        self.source_origin = source_origin

        self.origin_det = origin_det



        # create_proj_geom('cone', detector_spacing_x, detector_spacing_y, det_row_count, 
        #                   det_col_count, angles, source_origin, source_det)

        self.proj_geom = astra.create_proj_geom(self.proj_mode, 

                                                self.det_width, self.det_width,

                                                self.n_det_rows, self.n_det_cols,

                                                self.angles, 

                                                self.source_origin / self.det_width, # 1024

                                                self.origin_det / self.det_width)  # 0
        # 後兩個參數在cone_proj程式中分別是1024跟0
        
        self.proj_id   = astra.data3d.create('-sino', self.proj_geom, data = proj)



        # Available algorithms:

        # 'FDK_CUDA', 'SIRT3D_CUDA', 'CGLS3D_CUDA'

        self.algo = algo   

        self.iterations = iterations          

        self.alg_cfg = astra.astra_dict(self.algo)

        self.alg_cfg['ProjectionDataId'] = self.proj_id

        self.alg_cfg['ReconstructionDataId'] = self.vol_id

        self.alg_id = astra.algorithm.create(self.alg_cfg)  

        



    @classmethod

    def createf(cls, vol_size, n_angles = 720, algo = 'SIRT3D_CUDA', iterations = 1000):

        ang_range = np.pi * 2       

        vol_shape = (vol_size, vol_size, vol_size)

        det_row_count = int(vol_size * 2)  

        det_col_count = int(vol_size * 2)   

        source_origin = int(vol_size * 2.05)

        origin_det = int(vol_size * 1.)

        return cls(vol_shape = vol_shape, proj_shape = (det_row_count, n_angles, det_col_count), 

                 scan_range = (0, ang_range), 

                 angles = np.linspace(0,  ang_range, num = n_angles, endpoint = False), 

                 det_width = 1.0, source_origin = source_origin, origin_det = origin_det,

                 algo = algo, iterations = iterations)



    def create512f(cls, n_angles = 720, algo = 'SIRT3D_CUDA', iterations = 1000):

        return cls.createf(vol_size = 512, n_angles = n_angles, algo = algo, iterations = iterations)

    @classmethod

    def create256f(cls, n_angles = 360, algo = 'SIRT3D_CUDA', iterations = 1000):

        return cls.createf(vol_size = 256, n_angles = n_angles, algo = algo, iterations = iterations)                

    @classmethod

    def create128f(cls, n_angles = 180, algo = 'SIRT3D_CUDA', iterations = 1000):

        return cls.createf(vol_size = 128, n_angles = n_angles, algo = algo, iterations = iterations)



    def project(self, volume = None, keep_id = False):

        if not (volume is None):

            astra.data3d.store(self.vol_id, volume)  

        proj_id, proj = astra.creators.create_sino3d_gpu(self.vol_id, self.proj_geom, self.vol_geom)

        if keep_id:

            return proj_id, proj

        else:

            proj = np.array(proj) # (slices, angles, detectors)

            astra.data3d.delete(proj_id)

            return proj



    # Projection shapes: (slices, angles, detectors)

    def reconstruct(self, proj = None):

        if not(proj is None):       

            astra.data3d.store(self.proj_id, proj)       

        astra.algorithm.run(self.alg_id, self.iterations)

        rec = astra.data3d.get(self.vol_id)  
        
        # print(proj.shape)

        return rec



    # Destructor

    def release(self):

        astra.data3d.delete(self.vol_id)

        astra.data3d.delete(self.proj_id)

        astra.algorithm.delete(self.alg_id) 

def normalizeRange(A, source_min, source_d, target_min = 0.0, target_d = 1.0): 
    
    B = (A - source_min) / source_d * target_d + target_min
    
    return B    


def normalize(A, minimum = 0.0, maximum = 1.0): 
    
    mini = np.min(A)
    maxi = np.max(A)
    #B = (A - mini) / (maxi - mini) * (maximum - minimum) + minimum
    #return B
    
    return normalizeRange(A, mini, maxi - mini, minimum, maximum - minimum)


def showImage(image, size = None, figsize = None, cmap = 'gray'):
    
    plt.figure(figsize=figsize)
    
    if size is not None:
        image = np.reshape(image, size)
        
    if len(image.shape) > 2:
        
        nH, nW, nC = image.shape
        
        if nC == 1:
            image = np.reshape(image, image.shape[0:2])
            
    else:
        nH, nW = image.shape
        nC = 1

    if nC == 1:     
        plt.imshow(image, cmap=cmap, vmin = 0, vmax = 1)
    else:
        plt.imshow(image, vmin = 0, vmax = 1)
        
    plt.show() 

    
def parsePath(filepath):  
    p = Path(filepath)  
    directory = str(p.parent)
    filename = str(p.stem)
    extname = str(p.suffix)
    return directory, filename, extname

In [ ]:
# import and functions
# Cone.py 
# cone-beam tomography 
# (c) 2022, Chang-Chieh Cheng, jameschengcs@nycu.edu.tw


import numpy as np
import copy
import sys
sys.path.append('../')
#import vgi
import astra
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import scipy.ndimage
import torch
import torch.nn.functional as F


__all__ = ('ConeRec', 'astraProjShape', )

 
 # From (views, slices, detectors) = (slices, views, detectors)

def astraProjShape(proj):

    return np.swapaxes(proj, 0, 1)    



# ----------------------------------------------------

# Projection shapes: (slices, angles, detectors)

class ConeRec:

    def __init__(self, vol_shape, proj_shape, scan_range = (0, 2 * np.pi), angles = None, volume = None, proj = None,

                 det_width = 1.0, source_origin = 512., origin_det = 512.,

                 algo = 'SIRT3D_CUDA', iterations = 1000):

        self.vol_shape = vol_shape      # (d, h, w)

        self.depth, self.height, self.width = self.vol_shape

        self.proj_shape = proj_shape    # (slices, angles, detectors)

        self.n_det_rows, self.n_angles, self.n_det_cols = self.proj_shape

        self.scan_range = scan_range

        self.proj_mode = 'cone'      

        # create_vol_geom(Y, X, Z)``:  

        self.vol_geom = astra.create_vol_geom(self.height, self.width, self.depth)

        self.vol_id = astra.data3d.create('-vol', self.vol_geom, data = volume)

        if angles is None:

            self.angles = np.linspace(self.scan_range[0], self.scan_range[1], self.n_angles, False)

        else:

            self.angles = angles

        self.det_width = det_width

        self.source_origin = source_origin

        self.origin_det = origin_det



        # create_proj_geom('cone', detector_spacing_x, detector_spacing_y, det_row_count, 
        #                   det_col_count, angles, source_origin, source_det)

        self.proj_geom = astra.create_proj_geom(self.proj_mode, 

                                                self.det_width, self.det_width,

                                                self.n_det_rows, self.n_det_cols,

                                                self.angles, 

                                                self.source_origin / self.det_width, # 1024

                                                self.origin_det / self.det_width)  # 0
        # 後兩個參數在cone_proj程式中分別是1024跟0
        
        self.proj_id   = astra.data3d.create('-sino', self.proj_geom, data = proj)



        # Available algorithms:

        # 'FDK_CUDA', 'SIRT3D_CUDA', 'CGLS3D_CUDA'

        self.algo = algo   

        self.iterations = iterations          

        self.alg_cfg = astra.astra_dict(self.algo)

        self.alg_cfg['ProjectionDataId'] = self.proj_id

        self.alg_cfg['ReconstructionDataId'] = self.vol_id

        self.alg_id = astra.algorithm.create(self.alg_cfg)  

        



    @classmethod

    def createf(cls, vol_size, n_angles = 720, algo = 'SIRT3D_CUDA', iterations = 1000):

        ang_range = np.pi * 2       

        vol_shape = (vol_size, vol_size, vol_size)

        det_row_count = int(vol_size * 2)  

        det_col_count = int(vol_size * 2)   

        source_origin = int(vol_size * 2.05)

        origin_det = int(vol_size * 1.)

        return cls(vol_shape = vol_shape, proj_shape = (det_row_count, n_angles, det_col_count), 

                 scan_range = (0, ang_range), 

                 angles = np.linspace(0,  ang_range, num = n_angles, endpoint = False), 

                 det_width = 1.0, source_origin = source_origin, origin_det = origin_det,

                 algo = algo, iterations = iterations)



    def create512f(cls, n_angles = 720, algo = 'SIRT3D_CUDA', iterations = 1000):

        return cls.createf(vol_size = 512, n_angles = n_angles, algo = algo, iterations = iterations)

    @classmethod

    def create256f(cls, n_angles = 360, algo = 'SIRT3D_CUDA', iterations = 1000):

        return cls.createf(vol_size = 256, n_angles = n_angles, algo = algo, iterations = iterations)                

    @classmethod

    def create128f(cls, n_angles = 180, algo = 'SIRT3D_CUDA', iterations = 1000):

        return cls.createf(vol_size = 128, n_angles = n_angles, algo = algo, iterations = iterations)



    def project(self, volume = None, keep_id = False):

        if not (volume is None):

            astra.data3d.store(self.vol_id, volume)  

        proj_id, proj = astra.creators.create_sino3d_gpu(self.vol_id, self.proj_geom, self.vol_geom)

        if keep_id:

            return proj_id, proj

        else:

            proj = np.array(proj) # (slices, angles, detectors)

            astra.data3d.delete(proj_id)

            return proj



    # Projection shapes: (slices, angles, detectors)

    def reconstruct(self, proj = None):

        if not(proj is None):       

            astra.data3d.store(self.proj_id, proj)       

        astra.algorithm.run(self.alg_id, self.iterations)

        rec = astra.data3d.get(self.vol_id)  
        
        # print(proj.shape)

        return rec



    # Destructor

    def release(self):

        astra.data3d.delete(self.vol_id)

        astra.data3d.delete(self.proj_id)

        astra.algorithm.delete(self.alg_id) 

def normalizeRange(A, source_min, source_d, target_min = 0.0, target_d = 1.0): 
    
    B = (A - source_min) / source_d * target_d + target_min
    
    return B    


def normalize(A, minimum = 0.0, maximum = 1.0): 
    
    mini = np.min(A)
    maxi = np.max(A)
    #B = (A - mini) / (maxi - mini) * (maximum - minimum) + minimum
    #return B
    
    return normalizeRange(A, mini, maxi - mini, minimum, maximum - minimum)


def showImage(image, size = None, figsize = None, cmap = 'gray'):
    
    plt.figure(figsize=figsize)
    
    if size is not None:
        image = np.reshape(image, size)
        
    if len(image.shape) > 2:
        
        nH, nW, nC = image.shape
        
        if nC == 1:
            image = np.reshape(image, image.shape[0:2])
            
    else:
        nH, nW = image.shape
        nC = 1

    if nC == 1:     
        plt.imshow(image, cmap=cmap, vmin = 0, vmax = 1)
    else:
        plt.imshow(image, vmin = 0, vmax = 1)
        
    plt.show() 

    
def parsePath(filepath):  
    p = Path(filepath)  
    directory = str(p.parent)
    filename = str(p.stem)
    extname = str(p.suffix)
    return directory, filename, extname

def normalizeRange(A, source_min, source_d, target_min = 0.0, target_d = 1.0): 
    
    B = (A - source_min) / source_d * target_d + target_min
    
    return B    


def normalize(A, minimum = 0.0, maximum = 1.0): 
    
    mini = np.min(A)
    maxi = np.max(A)
    #B = (A - mini) / (maxi - mini) * (maximum - minimum) + minimum
    #return B
    
    return normalizeRange(A, mini, maxi - mini, minimum, maximum - minimum)


def showImage(image, size = None, figsize = None, cmap = 'gray'):
    
    plt.figure(figsize=figsize)
    
    if size is not None:
        image = np.reshape(image, size)
        
    if len(image.shape) > 2:
        
        nH, nW, nC = image.shape
        
        if nC == 1:
            image = np.reshape(image, image.shape[0:2])
            
    else:
        nH, nW = image.shape
        nC = 1

    if nC == 1:     
        plt.imshow(image, cmap=cmap, vmin = 0, vmax = 1)
    else:
        plt.imshow(image, vmin = 0, vmax = 1)
        
    plt.show() 

    
def parsePath(filepath):  
    p = Path(filepath)  
    directory = str(p.parent)
    filename = str(p.stem)
    extname = str(p.suffix)
    return directory, filename, extname

    
def interpolate(input_tensor, input_angle=30, output_angle=360):
    
    # target_shape = (input_tensor.size(0), output_angle, input_tensor.size(2))
    
    #print(input_tensor.shape)
    
    input_tensor = input_tensor.permute(0, 2, 1)
    
    #print(input_tensor.shape)
    
    output_tensor = F.interpolate(input_tensor, size=(output_angle), mode='linear')
    
    output_tensor = output_tensor.permute(0, 2, 1)
    
    return torch.squeeze(output_tensor)

def patch_algo_loop(total_length, patch_size, overlap_length):
    res = [0]
    i = 0
    
    while i < total_length:    
        if (total_length - (i + patch_size)) / patch_size <= 1:
            i += patch_size - (i + patch_size + patch_size - total_length)
            res.append(i)
            break
        else:
            i += patch_size - overlap_length
            res.append(i)
        
    return res

def show_tensor_distribution(data_tensor):
    max_value = torch.max(data_tensor)
    min_value = torch.min(data_tensor)
    print("Max value:", max_value)
    print("Min value:", min_value)

    data_array = data_tensor.cpu().detach().numpy().flatten()  
    plt.hist(data_array, bins=50)
    plt.title('Data Distribution')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
    plt.show()

    mean = torch.mean(data_tensor)
    std = torch.std(data_tensor)
    threshold = mean + 3 * std 
    outliers = torch.abs(data_tensor - mean) > threshold
    print("Outliers:", data_tensor[outliers])

    nan_mask = torch.isnan(data_tensor)
    nan_count = torch.sum(nan_mask).item()
    print("Number of NaN values:", nan_count)
    
def create_folder_if_not_exist(folder_path):
    
    if not os.path.exists(folder_path):
        os.makedirs(folder_path)
        print(f"makedirs {folder_path}")
    else:
        print(f"{folder_path} existed")

def test_model():
    model.eval()
    epochs = 1
    
    with torch.no_grad():
        for epoch in range(epochs):
            start_time = time.time()
            
            loss_sum = 0
            data_tensor = torch.zeros(n, y, z).to(device)
            weight_tensor = torch.zeros(n, y, z).to(device)
    
            for i in range(len(index_x)):
                for j in range(len(index_y)):
                    loss_value = 0
                    print('test:', 'i:', i, 'j:', j)
                    for k in range(len(index_z)):
                        
                        data = data_sinogram[index_x[i]:index_x[i]+patch_size_[0], index_y[j]:index_y[j]+patch_size_[1], index_z[k]:index_z[k]+patch_size_[2]]
                        data = data.unsqueeze(0).unsqueeze(0)
                        target = target_sinogram[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]]
                        target = target.unsqueeze(0).unsqueeze(0) 
    
                        output = model(data).to(device)
    
                        target = target.squeeze(0).squeeze(0)
                        output = output.squeeze(0).squeeze(0).squeeze(0)
    
                        output_temp = output.detach().cpu().numpy()
                        output_temp = copy.deepcopy(output_temp)
                        output_temp = torch.tensor(output_temp).to(device)
                        output_temp = copy.deepcopy(output_temp)
                        data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]].detach().cpu().numpy()
                        data_tensor_temp = copy.deepcopy(data_tensor_temp)
                        data_tensor_temp = torch.tensor(data_tensor_temp).to(device)
                        output += data_tensor_temp
                        
                        data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( output_temp )
                        weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( tensor_ones )
                        
                        output /= weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]]
                        loss = loss_fn(output, target)
                        #optimizer.zero_grad()
                        #loss.backward() 
                        #optimizer.step()
                        loss_value += loss.item()
                            
                    print('loss:', loss_value)
                    loss_sum += loss_value
                    print('loss_sum:', loss_sum)              
    
            print('final loss_sum:', loss_sum)
            losses.append(loss_sum)
    
            #torch.save(model.state_dict(), save_model)
            
            #create_folder_if_not_exist(data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram_output' % (target_size[1], target_size[2]))
            
            save_sino = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram_output' % (target_size[1], target_size[2])
            if not os.path.exists(save_sino + '/test_result'):
                os.mkdir(save_sino + '/test_result')
    
            data_tensor = data_tensor / weight_tensor
    
            for i in range(n):
                
                path = save_sino + '/test_result' + '/tomo_' + str("{:05d}".format(rand_start + i)) + '.tif'
                tifffile.imsave(path, np.array(data_tensor[i].cpu().detach().numpy()))

                # save in the next size folder
                path = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram/' % (target_size[1], target_size[2]) + '/tomo_' + str("{:05d}".format(rand_start + i)) + '.tif'
                tifffile.imsave(path, np.array(data_tensor[i].cpu().detach().numpy()))    
    
            print(data_tensor.shape)
            show_tensor_distribution(data_tensor)
            
            end_time = time.time()
            times.append(end_time - start_time)
            print(f"本epoch運行 {end_time - start_time} 秒")


def calculate_3d_psnr_ssim(image1, image2):
    
    # 3D PSNR
    psnr_value = psnr(image1, image2, data_range=image1.max() - image1.min())

    # 3D SSIM
    ssim_value = ssim(image1, image2, data_range=image1.max() - image1.min(), channel_axis=None)

    return psnr_value, ssim_value

# interpolate projections from 800 to 1201
def interpolate_process(recon_result, before_view = 800, after_view = 1201):

    zoom_factors = (1, after_view / before_view, 1)
    recon_result = scipy.ndimage.zoom(recon_result, zoom_factors, order=3)
    
    print(recon_result.shape)
    print(np.max(recon_result))
    print(np.min(recon_result))
    print(np.mean(recon_result))
    
    return recon_result

# load projection and downsample to 50 then upsample to 1201 by torch
def interpolate_torch(im, before_view = 1201, after_view = 50):
    # Step 1: Convert the NumPy array to a Torch tensor
    im_torch = torch.tensor(im, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # Add batch and channel dimensions

    # Step 2: Define the target size for interpolation
    # Assuming downsampling the second dimension to 150
    target_size = (im_torch.shape[2], after_view, im_torch.shape[4])

    # Step 3: Perform interpolation using Torch
    im_interpolated = F.interpolate(im_torch, size=target_size, mode='trilinear', align_corners=False)

    # Step 4: Remove the added dimensions to return to original shape
    im_interpolated = im_interpolated.squeeze(0).squeeze(0)

    # Convert back to NumPy array if needed
    im_interpolated_np = im_interpolated.numpy()
    
    return im_interpolated_np

In [ ]:
# MSNET multi scale

#============================================Projection-Domain U-Net==================================================
import random
import tifffile
import os
from skimage.metrics import structural_similarity as ssim
import copy
import matplotlib.pyplot as plt
import time
import scipy.ndimage
import gc


learning_rate = 0.001
learning_rate_decay = 1
epochs = 30 # 100 / last size 20
#rand_start = 660
data_list = ['L004.npy', 'L006.npy', 'L014.npy', 'L019.npy', 'C002.npy', 'C004.npy', 'C012.npy']#['C002.npy', 'C004.npy', 'C012.npy', 'L004.npy', 'L006.npy', 'L014.npy', 'L019.npy']#, 'C002_proj.npy', 'C004_proj.npy', 'C009_proj.npy', 'C012_proj.npy'] #, 'N005_proj.npy', 'N012_proj.npy', 'N021_proj.npy', 'N030_proj.npy', 'N047_proj.npy', 'N051_proj.npy', 'N053_proj.npy', 'N056_proj.npy', 'N072_proj.npy', 'N076_proj.npy', 'N078_proj.npy', 'N079_proj.npy', 'N082_proj.npy', 'N085_proj.npy', 'N090_proj.npy', 'N096_proj.npy', 'N100_proj.npy', 'N105_proj.npy', 'N181_proj.npy']
#['N001_proj.npy', 'N003_proj.npy', 'N005_proj.npy', 'N012_proj.npy', 'N021_proj.npy', 'N030_proj.npy', 'N047_proj.npy', 'N051_proj.npy', 'N053_proj.npy', 'N056_proj.npy', 'N072_proj.npy', 'N076_proj.npy', 'N078_proj.npy', 'N079_proj.npy', 'N082_proj.npy', 'N085_proj.npy', 'N090_proj.npy', 'N096_proj.npy', 'N100_proj.npy', 'N105_proj.npy', 'N181_proj.npy']
#data_list = ['L006_proj.npy', 'L012_proj.npy', 'L014_proj.npy', 'L019_proj.npy', 'L024_proj.npy', 'L027_proj.npy', 'L030_proj.npy', 'L033_proj.npy', 'L035_proj.npy', 'L036_proj.npy', 'L043_proj.npy', 'L044_proj.npy', 'L045_proj.npy', 'L048_proj.npy', 'L049_proj.npy', 'L056_proj.npy', 'L071_proj.npy', 'L133_proj.npy', 'L181_proj.npy', 'L231_proj.npy']
#[1]#,6,7,8,9,10,11,12,13,14,15,16]#['2seq_8','2seq_9','2seq_10','2seq_14']#, '2seq_10']
#rand_start_list = [660]#[147,569,1211,931]#, 1211]
#data_prefix_path = '2seq_13'
scale = 'multi_scale' # 'single_scale' 'multi_scale'
mode = 'image_domain' # 'proj_domain' 'image_domain'
test_mode = True #True
last_size = False
views = 100
full_views = 800#768
data_locate = '/media/jamescheng/My Passport/LDCT/proj_nml_vol/'
#model_locate = '/media/jamescheng/My Book/unet_model/'
#save_model = '/media/jamescheng/My Passport/unet_model/ldct/model_L.pth'
is_save_model = True
is_load_model = True

device = torch.device("cpu")
if torch.cuda.is_available(): device = torch.device("cuda")
print(device)

#target_size_list = [(1256, 200, 2480), (1256, 400, 2480), (1256, 800, 2480)]#, (768, 1200, 972)]#, (768, 1201, 972)]#, (768, 400, 972), (768, 800, 972), (768, 1201, 972)]#(2160, 128, 256), (2160, 350, 256), (2160, 350, 512), (2160, 800, 512), (2160, 800, 1024), (2160, 1817, 1024)]
for data_id in data_list:
    
    text = data_id[0]
    print(text)
    save_model = f'/media/jamescheng/My Passport/unet_model/ldct/model_{text}.pth'
    
    #save_model = f'/media/jamescheng/My Passport/unet_model/msnet_1015/msnet.pth'
    
    #first_time = True
    training_tensor = None
    
    if mode == 'proj_domain':
        projection_gt = np.load(f'{data_locate}{data_id}')
        #projection_gt = projection_gt.transpose(2, 1, 0)
        #zoom_factors = (1, views / full_views, 1)
        #projection = scipy.ndimage.zoom(projection_gt, zoom_factors, order=3)
        #projection = interpolate_torch(projection_gt, full_views, views)
        #print('current id: ', data_id)
        #print('projection: ', projection.shape)
        print('projection_gt: ', projection_gt.shape)
    elif mode == 'image_domain':
        projection_gt = np.load(f'/media/jamescheng/My Passport/LDCT/volume/{data_id}')
        print('projection_gt: ', projection_gt.shape)
    
    
    target_size_list = [(projection_gt.shape[0], 200, projection_gt.shape[2]), (projection_gt.shape[0], 400, projection_gt.shape[2]), (projection_gt.shape[0], 800, projection_gt.shape[2])]
    #target_size_list = target_size_list[2:]
    
    if mode == 'proj_domain':
        del projection_gt
        gc.collect()  # 强制进行垃圾回收
        torch.cuda.empty_cache()
    if mode == 'image_domain':
        target_size_list = [(projection_gt.shape[0], projection_gt.shape[1], projection_gt.shape[2])]
        
    for target_size in target_size_list:
        
        #!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        if target_size[1] == 200 and mode == 'proj_domain':
            save_model = f'/media/jamescheng/My Passport/unet_model/ldct/model_{text}1.pth'
            data_path = f'/media/jamescheng/My Passport/LDCT/result/proj100/{data_id}_sinogram.npy'
        if target_size[1] == 400 and mode == 'proj_domain':
            save_model = f'/media/jamescheng/My Passport/unet_model/ldct/model_{text}2.pth'
            data_path = f'/media/jamescheng/My Passport/LDCT/result/proj200_output/{data_id}_sinogram.npy'
        if target_size[1] == 800 and mode == 'proj_domain':
            save_model = f'/media/jamescheng/My Passport/unet_model/ldct/model_{text}3.pth'
            data_path = f'/media/jamescheng/My Passport/LDCT/result/proj400_output/{data_id}_sinogram.npy'
        if mode == 'image_domain':
            save_model = f'/media/jamescheng/My Passport/unet_model/ldct/msnet_{text}_image_domain.pth'
            data_path = f'/media/jamescheng/My Passport/LDCT/result/image_msnet/{data_id}'
        #!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        
        
        projection = np.load(data_path)
        print('current id: ', data_id)
        print('projection: ', projection.shape)
        #print('projection_gt: ', projection_gt.shape)
        
        target_path = f'/media/jamescheng/My Passport/LDCT/result/proj{target_size[1]}/{data_id}_sinogram.npy'
        if target_size[1] == 800:
            target_path = f'{data_locate}{data_id}'
        if mode == 'image_domain':
            target_path = f'/media/jamescheng/My Passport/LDCT/volume/{data_id}'
        #projection_gt = np.load(f'{data_locate}{data_id}')
        #projection_gt = projection_gt.transpose(2, 1, 0)
    
        #zoom_factors = (1, views / full_views, 1)
        #projection = scipy.ndimage.zoom(projection_gt, zoom_factors, order=3)
        #projection_target = interpolate_torch(projection_gt, full_views, target_size[1])
        projection_target = np.load(target_path)
        
        #del projection_gt
        gc.collect()  # 强制进行垃圾回收
        torch.cuda.empty_cache()
        
        print('target_size: ', target_size)
        print('projection_target_size: ', projection_target.shape)

        losses = []
        times = []
        #data_size = (512, target_size_list[1] / 2, 512)

        data_size = (target_size[0], views, target_size[2])
        if target_size_list[0][1] == 800:
            data_size = (target_size[0], 800, target_size[2])
            #last_size = True
        else:
            data_size = (target_size[0], target_size[1] // 2, target_size[2])

        #file_len = 2160
        #angle = 20
        #target_angle = file_len
        #data_folder = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram/' % (data_size[1], data_size[2])
        #target_folder = data_locate + data_prefix_path + '/proj128_multi_scale/target/trilinear%d_%d_sinogram/' %(target_size[1], target_size[2])
        #if target_size == (2160, 1817, 2560):
        #    target_folder = data_locate + data_prefix_path + '/sinogram/'


        # multi scale last size
        if last_size:
            patch_size_ = (32, 32, 32)
            overlap_length = (16, 16, 16)
            output_patch_size = (32, 64, 32)
            output_overlap_length = (16, 32, 16)

        # multi scale
        if not last_size:
            patch_size_ = (32, 32, 32)
            overlap_length = (16, 16, 16)
            output_patch_size = (32, 64, 32)
            output_overlap_length = (16, 32, 16)
            
        if mode == 'image_domain':
            patch_size_ = (32, 32, 32)
            overlap_length = (16, 16, 16)
            output_patch_size = (32, 32, 32)
            output_overlap_length = (16, 16, 16)
            
        tensor_ones = torch.ones(output_patch_size)#.to(device)

        #x = 512
        y = target_size[1]
        z = target_size[2]
        y_data = data_size[1]
        z_data = data_size[2] 
        #n = 100

        #output_proj = []
        #name_list = []
        #data_num = 0

        

        '''
        data_files = sorted(os.listdir(data_folder))
        target_files = sorted(os.listdir(target_folder))
        if data_size[1] == 128:
            data_files = data_files[rand_start : rand_start + n]
        target_files = target_files[rand_start : rand_start + n]

        print('data_folder len:', len(data_files))
        print('target_folder len:', len(target_files))   

        data_sinogram = torch.zeros(0, y_data, z_data).to(device)
        target_sinogram = torch.zeros(0, y, z).to(device)
        data_tensor = torch.zeros(n, y, z).to(device)
        weight_tensor = torch.zeros(n, y, z).to(device)

        for data_file, target_file in zip(data_files, target_files):

            data_path = os.path.join(data_folder, data_file)    
            target_path = os.path.join(target_folder, target_file)
            print('data_path', data_path)
            print('target_path', target_path)

            im = Image.open(data_path)    
            imarray = np.array(im)
            im_tensor = torch.from_numpy(imarray).to(device)
            im_tensor = im_tensor.unsqueeze(0)
            data_sinogram = torch.cat((data_sinogram, im_tensor), dim=0)
            print('data_sinogram shape', data_sinogram.shape)

            im = Image.open(target_path)    
            imarray = np.array(im)
            im_tensor = torch.from_numpy(imarray).to(device)
            im_tensor = im_tensor.unsqueeze(0)
            target_sinogram = torch.cat((target_sinogram, im_tensor), dim=0)
            print('target_sinogram shape', target_sinogram.shape)

            directory, filename, extname = parsePath(data_path)
            filename = filename + '.tif'
            name_list.append(filename)
        '''
        
        #if first_time == False:
        #    data_sinogram = training_tensor
        #else:
        #    data_sinogram = torch.from_numpy(projection).to(device)
        data_sinogram = torch.from_numpy(projection).to(device, dtype=torch.float32)
        
        target_sinogram = torch.from_numpy(projection_target).to(device, dtype=torch.float32)
        print(type(data_sinogram))
        print(data_sinogram.shape)
        print(type(target_sinogram))
        print(target_sinogram.shape)
        
        del projection_target
        del projection
        gc.collect()  # 强制进行垃圾回收
        torch.cuda.empty_cache()

        index_x = patch_algo_loop(data_sinogram.shape[0], patch_size_[0], overlap_length[0])
        index_y = patch_algo_loop(data_sinogram.shape[1], patch_size_[1], overlap_length[1])
        index_z = patch_algo_loop(data_sinogram.shape[2], patch_size_[2], overlap_length[2])

        print(index_x)
        print(index_y)
        print(index_z)

        index_x_output = patch_algo_loop(target_sinogram.shape[0], output_patch_size[0], output_overlap_length[0])
        index_y_output = patch_algo_loop(target_sinogram.shape[1], output_patch_size[1], output_overlap_length[1])
        index_z_output = patch_algo_loop(target_sinogram.shape[2], output_patch_size[2], output_overlap_length[2])

        print(index_x_output)
        print(index_y_output)
        print(index_z_output)

        if test_mode:
            del target_sinogram
        
        #tensor_ones = torch.ones(output_patch_size).to(device)

        total_time = 0

        model = MSNet_multi().to(device)
        #model.load_state_dict(torch.load(save_model))
        loss_fn = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        if is_load_model:
            print(save_model)
            checkpoint = torch.load(save_model)
            model.load_state_dict(checkpoint['model_state_dict'], strict=False)
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])


        if test_mode:

            #test

            model.eval()
            epochs = 1

            with torch.no_grad():
                for epoch in range(epochs):
                    start_time = time.time()

                    loss_sum = 0
                    
                    '''
                    data_tensor = torch.zeros(target_size).to(device)
                    weight_tensor = torch.zeros(target_size).to(device)
                    '''
                    
                    data_tensor = torch.zeros(target_size)#.to(device)
                    weight_tensor = torch.zeros(target_size, dtype=torch.float32, requires_grad=False)#.to(device)
                    
                    for i in range(len(index_x)):
                        for j in range(len(index_y)):
                            loss_value = 0
                            print('test:', 'i:', i, 'j:', j)
                            for k in range(len(index_z)):

                                '''
                                data = data_sinogram[index_x[i]:index_x[i]+patch_size_[0], index_y[j]:index_y[j]+patch_size_[1], index_z[k]:index_z[k]+patch_size_[2]]
                                data = data.unsqueeze(0).unsqueeze(0).to(device)
                                #target = target_sinogram[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]]
                                #target = target.unsqueeze(0).unsqueeze(0) 

                                output = model(data).to(device)

                                #target = target.squeeze(0).squeeze(0)
                                output = output.squeeze(0).squeeze(0).squeeze(0).to(device)

                                output_temp = output.detach().cpu().numpy()
                                output_temp = copy.deepcopy(output_temp)
                                output_temp = torch.tensor(output_temp).to(device)
                                output_temp = copy.deepcopy(output_temp).to(device)
                                data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]].detach().cpu().numpy()
                                data_tensor_temp = copy.deepcopy(data_tensor_temp)
                                data_tensor_temp = torch.tensor(data_tensor_temp).to(device)
                                output += data_tensor_temp

                                data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( output_temp )
                                weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( tensor_ones.to(device) )

                                output /= weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]]
                                #loss = loss_fn(output, target)
                                #optimizer.zero_grad()
                                #loss.backward() 
                                #optimizer.step()
                                #loss_value += loss.item()
                                '''
                                #optimizer.param_groups[0]['lr'] = optimizer.param_groups[0]['lr'] * learning_rate_decay
                                #print('learning rate = ', optimizer.param_groups[0]['lr'])

                                data = data_sinogram[index_x[i]:index_x[i]+patch_size_[0], index_y[j]:index_y[j]+patch_size_[1], index_z[k]:index_z[k]+patch_size_[2]]
                                data = data.unsqueeze(0).unsqueeze(0)#.to(device)
                                #target = target_sinogram[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]]
                                #target = target.unsqueeze(0).unsqueeze(0)#.to(device)

                                output = model(data).to(torch.device("cpu"))

                                #target = target.squeeze(0).squeeze(0)#.to(device)
                                output = output.squeeze(0).squeeze(0).squeeze(0)#.to(device)

                                #output_temp = output.detach().cpu().numpy()
                                #output_temp = copy.deepcopy(output_temp)
                                #output_temp = torch.tensor(output_temp).to(device)
                                #output_temp = copy.deepcopy(output_temp)

                                output_temp = output.clone().detach()  # 只需要 clone 一次

                                #data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]].detach().cpu().numpy()
                                #data_tensor_temp = copy.deepcopy(data_tensor_temp)
                                #data_tensor_temp = torch.tensor(data_tensor_temp).to(device)

                                data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], 
                                   index_y_output[j]:index_y_output[j]+output_patch_size[1], 
                                   index_z_output[k]:index_z_output[k]+output_patch_size[2]].clone().detach()

                                output += data_tensor_temp

                                data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( output_temp )
                                weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( tensor_ones )

                                output.div_(weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]])

                                #loss = loss_fn(output, target.to(torch.device("cpu")))#.to(device)
                                #optimizer.zero_grad()
                                #loss.backward() 
                                #optimizer.step()

                                #loss_value += loss.detach().cpu().item()

                                #print('loss:', loss_value)
                                #loss_sum += loss_value
                                #print('loss_sum:', loss_sum)              

                    #print('final loss_sum:', loss_sum)
                    #losses.append(loss_sum)

                    #torch.save(model.state_dict(), save_model)

                    #create_folder_if_not_exist(data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram_output' % (target_size[1], target_size[2]))

                    #save_sino = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram_output' % (target_size[1], target_size[2])
                    #if not os.path.exists(save_sino + '/test_result'):
                        #os.mkdir(save_sino + '/test_result')
                    #    pass

                    data_tensor = data_tensor / weight_tensor
                    '''
                    for i in range(n):

                        path = save_sino + '/test_result' + '/tomo_' + str("{:05d}".format(rand_start + i)) + '.tif'
                        tifffile.imsave(path, np.array(data_tensor[i].cpu().detach().numpy()))

                        # save in the next size folder
                        path = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram/' % (target_size[1], target_size[2]) + '/tomo_' + str("{:05d}".format(rand_start + i)) + '.tif'
                        tifffile.imsave(path, np.array(data_tensor[i].cpu().detach().numpy()))    
                    '''
                    print(data_tensor.shape)
                    #show_tensor_distribution(data_tensor)

                    end_time = time.time()
                    times.append(end_time - start_time)
                    print(f"本epoch運行 {end_time - start_time} 秒")

        else:

            # train
            model.train()
            
            with torch.no_grad():
                data_tensor = torch.zeros(target_size, dtype=torch.float32)#.to(device)
                weight_tensor = torch.zeros(target_size, dtype=torch.float32, requires_grad=False)#.to(device)
                

            for epoch in range(epochs):
                start_time = time.time()

                loss_sum = 0
                #with torch.no_grad():
                #    data_tensor = torch.zeros(target_size).to(device)
                #    weight_tensor = torch.zeros(target_size, device=device, dtype=torch.float32, requires_grad=False)
                data_tensor.zero_()
                weight_tensor.zero_()
                
                for i in range(len(index_x)):
                    for j in range(len(index_y)):
                        loss_value = 0
                        print('epoch:', epoch, 'i:', i, 'j:', j)
                        #optimizer.param_groups[0]['lr'] = optimizer.param_groups[0]['lr'] * learning_rate_decay
                        #print('learning rate = ', optimizer.param_groups[0]['lr'])
                        for k in range(len(index_z)):
                            #optimizer.param_groups[0]['lr'] = optimizer.param_groups[0]['lr'] * learning_rate_decay
                            #print('learning rate = ', optimizer.param_groups[0]['lr'])

                            data = data_sinogram[index_x[i]:index_x[i]+patch_size_[0], index_y[j]:index_y[j]+patch_size_[1], index_z[k]:index_z[k]+patch_size_[2]]
                            data = data.unsqueeze(0).unsqueeze(0)#.to(device)
                            target = target_sinogram[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]]
                            target = target.unsqueeze(0).unsqueeze(0)#.to(device)

                            output = model(data).to(torch.device("cpu"))

                            target = target.squeeze(0).squeeze(0)#.to(device)
                            output = output.squeeze(0).squeeze(0).squeeze(0)#.to(device)

                            #output_temp = output.detach().cpu().numpy()
                            #output_temp = copy.deepcopy(output_temp)
                            #output_temp = torch.tensor(output_temp).to(device)
                            #output_temp = copy.deepcopy(output_temp)
                            
                            output_temp = output.clone().detach()  # 只需要 clone 一次
                            
                            #data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]].detach().cpu().numpy()
                            #data_tensor_temp = copy.deepcopy(data_tensor_temp)
                            #data_tensor_temp = torch.tensor(data_tensor_temp).to(device)
                            
                            data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], 
                               index_y_output[j]:index_y_output[j]+output_patch_size[1], 
                               index_z_output[k]:index_z_output[k]+output_patch_size[2]].clone().detach()

                            output += data_tensor_temp

                            data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( output_temp )
                            weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( tensor_ones )

                            output.div_(weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]])
                            
                            loss = loss_fn(output, target.to(torch.device("cpu")))#.to(device)
                            optimizer.zero_grad()
                            loss.backward() 
                            optimizer.step()
                            
                            loss_value += loss.detach().cpu().item()

                        print('loss:', loss_value)
                        loss_sum += loss_value
                        print('loss_sum:', loss_sum)              

                print('final loss_sum:', loss_sum)
                losses.append(loss_sum)

                #create_folder_if_not_exist(data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram_output' % (target_size[1], target_size[2]))

                #save_sino = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram_output' % (target_size[1], target_size[2])
                #if not os.path.exists(save_sino + '/' + str(loss_sum)):
                    #os.mkdir(save_sino + '/' + str(loss_sum))
                    #pass
                    
                with torch.no_grad():
                    data_tensor /= weight_tensor.clone().detach()
                    
                '''
                for i in range(n):

                    if epoch % 10 == 0:
                        path = save_sino + '/' + str(loss_sum) + '/tomo_' + str("{:05d}".format(rand_start + i)) + '.tif'
                        #tifffile.imsave(path, np.array(data_tensor[i].cpu().detach().numpy()))

                    # save in the next size folder
                    path = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram/' % (target_size[1], target_size[2]) + '/tomo_' + str("{:05d}".format(rand_start + i)) + '.tif'
                    #tifffile.imsave(path, np.array(data_tensor[i].cpu().detach().numpy()))          
                '''
                print(data_tensor.shape)
                #show_tensor_distribution(data_tensor)
                
                #show_slices(data_tensor.cpu().numpy()) # .clone().detach()

                end_time = time.time()
                times.append(end_time - start_time)
                print(f"本epoch運行 {end_time - start_time} 秒")

                if is_save_model:
                    torch.save(model.state_dict(), save_model)
                    torch.save({
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict()
                    }, save_model)
                
                #del weight_tensor
                #del data_sinogram
                #del target_sinogram
                gc.collect()  # 强制进行垃圾回收
                torch.cuda.empty_cache()
                
        if not test_mode:
            eps = list(range(1, len(losses) + 1))
            total_time = sum(times)

            plt.plot(eps, losses, label='Training Loss')
            plt.title('Training Loss Over Epochs')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            #plt.savefig(save_sino + '/Training_Loss.png')
            plt.show()

            plt.plot(eps, times, label='Training Time')
            plt.title('Training Time Over Epochs')
            plt.xlabel('Epoch')
            plt.ylabel('Time')
            plt.legend()
            #plt.savefig(save_sino + '/Training_Time.png')
            plt.show()

            total_params = sum(p.numel() for p in model.parameters())
            '''
            log_path = save_sino + '/log.txt'
            with open(log_path, "w") as file:
                if scale == 'multi_scale' and mode == 'proj_domain':
                    file.write('single scale sinogram training\n')
                    file.write('sinogram size: ' + str(data_size) + ' to ' + str(target_size_list[0]) + '\n')
                if scale == 'multi_scale' and mode == 'image_domain':
                    file.write('single scale recon training\n')
                    file.write('recon size: ' + str(target_size_list[0]) + ' to ' + str(target_size_list[0]) + '\n')

                file.write('patch_size: ' + str(patch_size_) + '\n')
                file.write('overlap_length: ' + str(overlap_length) + '\n')
                file.write('epoch: ' + str(len(losses)) + '\n')
                file.write('loss: ' + str(losses) +  '\n')
                file.write('time: ' +  str(total_time) +  '\n')
                file.write('parameters: ' + str(total_params) +  '\n')
            '''
            if scale == 'multi_scale' and mode == 'proj_domain':
                    print('single scale sinogram training')
                    print('sinogram size: ', str(data_size), ' to ', str(target_size))
            if scale == 'multi_scale' and mode == 'image_domain':
                    print('single scale recon training')
                    print('recon size: ', str(data_size), ' to ', str(target_size))   
            print('patch_size: ', str(patch_size_))
            print('overlap_length: ', str(overlap_length))
            print('epoch: ', str(len(losses)))
            print('loss: ', str(losses))
            print('time: ', str(total_time))
            print('parameters: ', str(total_params))
        
        #training_tensor = data_tensor
        #first_time = False
        if target_size[1] != 800 and mode == 'proj_domain':
            np.save(f'/media/jamescheng/My Passport/LDCT/result/proj{target_size[1]}_output/{data_id}_sinogram.npy', data_tensor.cpu().numpy())
        elif mode == 'proj_domain':
            np.save(f'/media/jamescheng/My Passport/LDCT/result/{data_id}_msnet_sinogram_result2.npy', data_tensor.cpu().numpy())
        elif mode == 'image_domain':
            np.save(f'/media/jamescheng/My Passport/LDCT/result/image_msnet/result/{data_id}_msnet_image_result.npy', data_tensor.cpu().numpy())
        
        del data_tensor
        del weight_tensor
        del data_sinogram
        #del target_sinogram
        gc.collect()  # 强制进行垃圾回收
        torch.cuda.empty_cache()
        
    #res = training_tensor.cpu().numpy()
    #res = interpolate_process(res, 800, 1201)
    #np.save(f'/media/jamescheng/My Passport/LDCT/result/{data_id}_msnet_sinogram_result.npy', training_tensor.cpu().numpy())
    
# multi scale proj-domain train test

# multi scale前段的size

In [ ]:
# HDNET and HDNET+

#============================================Projection-Domain U-Net==================================================
import random
import tifffile
import os
from skimage.metrics import structural_similarity as ssim
import copy
import matplotlib.pyplot as plt
import time
import scipy.ndimage
import gc


learning_rate = 0.001
learning_rate_decay = 1
epochs = 30 # 100 / last size 20
#rand_start = 660
data_list = ['L004.npy', 'L006.npy', 'L014.npy', 'L019.npy', 'C002.npy', 'C004.npy', 'C012.npy']#['C002.npy', 'C004.npy', 'C012.npy', 'L004.npy', 'L006.npy', 'L014.npy', 'L019.npy']#, 'C002_proj.npy', 'C004_proj.npy', 'C009_proj.npy', 'C012_proj.npy'] #, 'N005_proj.npy', 'N012_proj.npy', 'N021_proj.npy', 'N030_proj.npy', 'N047_proj.npy', 'N051_proj.npy', 'N053_proj.npy', 'N056_proj.npy', 'N072_proj.npy', 'N076_proj.npy', 'N078_proj.npy', 'N079_proj.npy', 'N082_proj.npy', 'N085_proj.npy', 'N090_proj.npy', 'N096_proj.npy', 'N100_proj.npy', 'N105_proj.npy', 'N181_proj.npy']
#['N001_proj.npy', 'N003_proj.npy', 'N005_proj.npy', 'N012_proj.npy', 'N021_proj.npy', 'N030_proj.npy', 'N047_proj.npy', 'N051_proj.npy', 'N053_proj.npy', 'N056_proj.npy', 'N072_proj.npy', 'N076_proj.npy', 'N078_proj.npy', 'N079_proj.npy', 'N082_proj.npy', 'N085_proj.npy', 'N090_proj.npy', 'N096_proj.npy', 'N100_proj.npy', 'N105_proj.npy', 'N181_proj.npy']
#data_list = ['L006_proj.npy', 'L012_proj.npy', 'L014_proj.npy', 'L019_proj.npy', 'L024_proj.npy', 'L027_proj.npy', 'L030_proj.npy', 'L033_proj.npy', 'L035_proj.npy', 'L036_proj.npy', 'L043_proj.npy', 'L044_proj.npy', 'L045_proj.npy', 'L048_proj.npy', 'L049_proj.npy', 'L056_proj.npy', 'L071_proj.npy', 'L133_proj.npy', 'L181_proj.npy', 'L231_proj.npy']
#[1]#,6,7,8,9,10,11,12,13,14,15,16]#['2seq_8','2seq_9','2seq_10','2seq_14']#, '2seq_10']
#rand_start_list = [660]#[147,569,1211,931]#, 1211]
#data_prefix_path = '2seq_13'
scale = 'multi_scale' # 'single_scale' 'multi_scale'
mode = 'image_domain' # 'proj_domain' 'image_domain'
test_mode = True #True
last_size = False
views = 100
full_views = 800#768
data_locate = '/media/jamescheng/My Passport/LDCT/proj_nml_vol/'
#model_locate = '/media/jamescheng/My Book/unet_model/'
#save_model = '/media/jamescheng/My Passport/unet_model/ldct/model_L.pth'
is_save_model = True
is_load_model = True

device = torch.device("cpu")
if torch.cuda.is_available(): device = torch.device("cuda")
print(device)

#target_size_list = [(1256, 200, 2480), (1256, 400, 2480), (1256, 800, 2480)]#, (768, 1200, 972)]#, (768, 1201, 972)]#, (768, 400, 972), (768, 800, 972), (768, 1201, 972)]#(2160, 128, 256), (2160, 350, 256), (2160, 350, 512), (2160, 800, 512), (2160, 800, 1024), (2160, 1817, 1024)]
for data_id in data_list:
    
    text = data_id[0]
    print(text)
    save_model = f'/media/jamescheng/My Passport/unet_model/ldct/model_{text}.pth'
    
    #save_model = f'/media/jamescheng/My Passport/unet_model/msnet_1015/msnet.pth'
    
    #first_time = True
    training_tensor = None
    
    if mode == 'proj_domain':
        projection_gt = np.load(f'{data_locate}{data_id}')
        #projection_gt = projection_gt.transpose(2, 1, 0)
        #zoom_factors = (1, views / full_views, 1)
        #projection = scipy.ndimage.zoom(projection_gt, zoom_factors, order=3)
        #projection = interpolate_torch(projection_gt, full_views, views)
        #print('current id: ', data_id)
        #print('projection: ', projection.shape)
        print('projection_gt: ', projection_gt.shape)
    elif mode == 'image_domain':
        projection_gt = np.load(f'/media/jamescheng/My Passport/LDCT/volume/{data_id}')
        print('projection_gt: ', projection_gt.shape)
    
    
    target_size_list = [(projection_gt.shape[0], 200, projection_gt.shape[2]), (projection_gt.shape[0], 400, projection_gt.shape[2]), (projection_gt.shape[0], 800, projection_gt.shape[2])]
    #target_size_list = target_size_list[2:]
    
    if mode == 'proj_domain':
        del projection_gt
        gc.collect()  # 强制进行垃圾回收
        torch.cuda.empty_cache()
    if mode == 'image_domain':
        target_size_list = [(projection_gt.shape[0], projection_gt.shape[1], projection_gt.shape[2])]
        
    for target_size in target_size_list:
        
        #!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        if target_size[1] == 200 and mode == 'proj_domain':
            save_model = f'/media/jamescheng/My Passport/unet_model/ldct/model_{text}1.pth'
            data_path = f'/media/jamescheng/My Passport/LDCT/result/proj100/{data_id}_sinogram.npy'
        if target_size[1] == 400 and mode == 'proj_domain':
            save_model = f'/media/jamescheng/My Passport/unet_model/ldct/model_{text}2.pth'
            data_path = f'/media/jamescheng/My Passport/LDCT/result/proj200_output/{data_id}_sinogram.npy'
        if target_size[1] == 800 and mode == 'proj_domain':
            save_model = f'/media/jamescheng/My Passport/unet_model/ldct/model_{text}3.pth'
            data_path = f'/media/jamescheng/My Passport/LDCT/result/proj400_output/{data_id}_sinogram.npy'
        if mode == 'image_domain':
            save_model = f'/media/jamescheng/My Passport/unet_model/ldct/msnet_{text}_image_domain.pth'
            data_path = f'/media/jamescheng/My Passport/LDCT/result/image_msnet/{data_id}'
        #!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        
        
        projection = np.load(data_path)
        print('current id: ', data_id)
        print('projection: ', projection.shape)
        #print('projection_gt: ', projection_gt.shape)
        
        target_path = f'/media/jamescheng/My Passport/LDCT/result/proj{target_size[1]}/{data_id}_sinogram.npy'
        if target_size[1] == 800:
            target_path = f'{data_locate}{data_id}'
        if mode == 'image_domain':
            target_path = f'/media/jamescheng/My Passport/LDCT/volume/{data_id}'
        #projection_gt = np.load(f'{data_locate}{data_id}')
        #projection_gt = projection_gt.transpose(2, 1, 0)
    
        #zoom_factors = (1, views / full_views, 1)
        #projection = scipy.ndimage.zoom(projection_gt, zoom_factors, order=3)
        #projection_target = interpolate_torch(projection_gt, full_views, target_size[1])
        projection_target = np.load(target_path)
        
        #del projection_gt
        gc.collect()  # 强制进行垃圾回收
        torch.cuda.empty_cache()
        
        print('target_size: ', target_size)
        print('projection_target_size: ', projection_target.shape)

        losses = []
        times = []
        #data_size = (512, target_size_list[1] / 2, 512)

        data_size = (target_size[0], views, target_size[2])
        if target_size_list[0][1] == 800:
            data_size = (target_size[0], 800, target_size[2])
            #last_size = True
        else:
            data_size = (target_size[0], target_size[1] // 2, target_size[2])

        #file_len = 2160
        #angle = 20
        #target_angle = file_len
        #data_folder = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram/' % (data_size[1], data_size[2])
        #target_folder = data_locate + data_prefix_path + '/proj128_multi_scale/target/trilinear%d_%d_sinogram/' %(target_size[1], target_size[2])
        #if target_size == (2160, 1817, 2560):
        #    target_folder = data_locate + data_prefix_path + '/sinogram/'


        # multi scale last size
        if last_size:
            patch_size_ = (32, 32, 32)
            overlap_length = (16, 16, 16)
            output_patch_size = (32, 64, 32)
            output_overlap_length = (16, 32, 16)

        # multi scale
        if not last_size:
            patch_size_ = (32, 32, 32)
            overlap_length = (16, 16, 16)
            output_patch_size = (32, 64, 32)
            output_overlap_length = (16, 32, 16)
            
        if mode == 'image_domain':
            patch_size_ = (32, 32, 32)
            overlap_length = (16, 16, 16)
            output_patch_size = (32, 32, 32)
            output_overlap_length = (16, 16, 16)
            
        tensor_ones = torch.ones(output_patch_size)#.to(device)

        #x = 512
        y = target_size[1]
        z = target_size[2]
        y_data = data_size[1]
        z_data = data_size[2] 
        #n = 100

        #output_proj = []
        #name_list = []
        #data_num = 0

        

        '''
        data_files = sorted(os.listdir(data_folder))
        target_files = sorted(os.listdir(target_folder))
        if data_size[1] == 128:
            data_files = data_files[rand_start : rand_start + n]
        target_files = target_files[rand_start : rand_start + n]

        print('data_folder len:', len(data_files))
        print('target_folder len:', len(target_files))   

        data_sinogram = torch.zeros(0, y_data, z_data).to(device)
        target_sinogram = torch.zeros(0, y, z).to(device)
        data_tensor = torch.zeros(n, y, z).to(device)
        weight_tensor = torch.zeros(n, y, z).to(device)

        for data_file, target_file in zip(data_files, target_files):

            data_path = os.path.join(data_folder, data_file)    
            target_path = os.path.join(target_folder, target_file)
            print('data_path', data_path)
            print('target_path', target_path)

            im = Image.open(data_path)    
            imarray = np.array(im)
            im_tensor = torch.from_numpy(imarray).to(device)
            im_tensor = im_tensor.unsqueeze(0)
            data_sinogram = torch.cat((data_sinogram, im_tensor), dim=0)
            print('data_sinogram shape', data_sinogram.shape)

            im = Image.open(target_path)    
            imarray = np.array(im)
            im_tensor = torch.from_numpy(imarray).to(device)
            im_tensor = im_tensor.unsqueeze(0)
            target_sinogram = torch.cat((target_sinogram, im_tensor), dim=0)
            print('target_sinogram shape', target_sinogram.shape)

            directory, filename, extname = parsePath(data_path)
            filename = filename + '.tif'
            name_list.append(filename)
        '''
        
        #if first_time == False:
        #    data_sinogram = training_tensor
        #else:
        #    data_sinogram = torch.from_numpy(projection).to(device)
        data_sinogram = torch.from_numpy(projection).to(device, dtype=torch.float32)
        
        target_sinogram = torch.from_numpy(projection_target).to(device, dtype=torch.float32)
        print(type(data_sinogram))
        print(data_sinogram.shape)
        print(type(target_sinogram))
        print(target_sinogram.shape)
        
        del projection_target
        del projection
        gc.collect()  # 强制进行垃圾回收
        torch.cuda.empty_cache()

        index_x = patch_algo_loop(data_sinogram.shape[0], patch_size_[0], overlap_length[0])
        index_y = patch_algo_loop(data_sinogram.shape[1], patch_size_[1], overlap_length[1])
        index_z = patch_algo_loop(data_sinogram.shape[2], patch_size_[2], overlap_length[2])

        print(index_x)
        print(index_y)
        print(index_z)

        index_x_output = patch_algo_loop(target_sinogram.shape[0], output_patch_size[0], output_overlap_length[0])
        index_y_output = patch_algo_loop(target_sinogram.shape[1], output_patch_size[1], output_overlap_length[1])
        index_z_output = patch_algo_loop(target_sinogram.shape[2], output_patch_size[2], output_overlap_length[2])

        print(index_x_output)
        print(index_y_output)
        print(index_z_output)

        if test_mode:
            del target_sinogram
        
        #tensor_ones = torch.ones(output_patch_size).to(device)

        total_time = 0

        model = MSNet_multi().to(device)
        #model.load_state_dict(torch.load(save_model))
        loss_fn = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)

        if is_load_model:
            print(save_model)
            checkpoint = torch.load(save_model)
            model.load_state_dict(checkpoint['model_state_dict'], strict=False)
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])


        if test_mode:

            #test

            model.eval()
            epochs = 1

            with torch.no_grad():
                for epoch in range(epochs):
                    start_time = time.time()

                    loss_sum = 0
                    
                    '''
                    data_tensor = torch.zeros(target_size).to(device)
                    weight_tensor = torch.zeros(target_size).to(device)
                    '''
                    
                    data_tensor = torch.zeros(target_size)#.to(device)
                    weight_tensor = torch.zeros(target_size, dtype=torch.float32, requires_grad=False)#.to(device)
                    
                    for i in range(len(index_x)):
                        for j in range(len(index_y)):
                            loss_value = 0
                            print('test:', 'i:', i, 'j:', j)
                            for k in range(len(index_z)):

                                '''
                                data = data_sinogram[index_x[i]:index_x[i]+patch_size_[0], index_y[j]:index_y[j]+patch_size_[1], index_z[k]:index_z[k]+patch_size_[2]]
                                data = data.unsqueeze(0).unsqueeze(0).to(device)
                                #target = target_sinogram[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]]
                                #target = target.unsqueeze(0).unsqueeze(0) 

                                output = model(data).to(device)

                                #target = target.squeeze(0).squeeze(0)
                                output = output.squeeze(0).squeeze(0).squeeze(0).to(device)

                                output_temp = output.detach().cpu().numpy()
                                output_temp = copy.deepcopy(output_temp)
                                output_temp = torch.tensor(output_temp).to(device)
                                output_temp = copy.deepcopy(output_temp).to(device)
                                data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]].detach().cpu().numpy()
                                data_tensor_temp = copy.deepcopy(data_tensor_temp)
                                data_tensor_temp = torch.tensor(data_tensor_temp).to(device)
                                output += data_tensor_temp

                                data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( output_temp )
                                weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( tensor_ones.to(device) )

                                output /= weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]]
                                #loss = loss_fn(output, target)
                                #optimizer.zero_grad()
                                #loss.backward() 
                                #optimizer.step()
                                #loss_value += loss.item()
                                '''
                                #optimizer.param_groups[0]['lr'] = optimizer.param_groups[0]['lr'] * learning_rate_decay
                                #print('learning rate = ', optimizer.param_groups[0]['lr'])

                                data = data_sinogram[index_x[i]:index_x[i]+patch_size_[0], index_y[j]:index_y[j]+patch_size_[1], index_z[k]:index_z[k]+patch_size_[2]]
                                data = data.unsqueeze(0).unsqueeze(0)#.to(device)
                                #target = target_sinogram[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]]
                                #target = target.unsqueeze(0).unsqueeze(0)#.to(device)

                                output = model(data).to(torch.device("cpu"))

                                #target = target.squeeze(0).squeeze(0)#.to(device)
                                output = output.squeeze(0).squeeze(0).squeeze(0)#.to(device)

                                #output_temp = output.detach().cpu().numpy()
                                #output_temp = copy.deepcopy(output_temp)
                                #output_temp = torch.tensor(output_temp).to(device)
                                #output_temp = copy.deepcopy(output_temp)

                                output_temp = output.clone().detach()  # 只需要 clone 一次

                                #data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]].detach().cpu().numpy()
                                #data_tensor_temp = copy.deepcopy(data_tensor_temp)
                                #data_tensor_temp = torch.tensor(data_tensor_temp).to(device)

                                data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], 
                                   index_y_output[j]:index_y_output[j]+output_patch_size[1], 
                                   index_z_output[k]:index_z_output[k]+output_patch_size[2]].clone().detach()

                                output += data_tensor_temp

                                data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( output_temp )
                                weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( tensor_ones )

                                output.div_(weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]])

                                #loss = loss_fn(output, target.to(torch.device("cpu")))#.to(device)
                                #optimizer.zero_grad()
                                #loss.backward() 
                                #optimizer.step()

                                #loss_value += loss.detach().cpu().item()

                                #print('loss:', loss_value)
                                #loss_sum += loss_value
                                #print('loss_sum:', loss_sum)              

                    #print('final loss_sum:', loss_sum)
                    #losses.append(loss_sum)

                    #torch.save(model.state_dict(), save_model)

                    #create_folder_if_not_exist(data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram_output' % (target_size[1], target_size[2]))

                    #save_sino = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram_output' % (target_size[1], target_size[2])
                    #if not os.path.exists(save_sino + '/test_result'):
                        #os.mkdir(save_sino + '/test_result')
                    #    pass

                    data_tensor = data_tensor / weight_tensor
                    '''
                    for i in range(n):

                        path = save_sino + '/test_result' + '/tomo_' + str("{:05d}".format(rand_start + i)) + '.tif'
                        tifffile.imsave(path, np.array(data_tensor[i].cpu().detach().numpy()))

                        # save in the next size folder
                        path = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram/' % (target_size[1], target_size[2]) + '/tomo_' + str("{:05d}".format(rand_start + i)) + '.tif'
                        tifffile.imsave(path, np.array(data_tensor[i].cpu().detach().numpy()))    
                    '''
                    print(data_tensor.shape)
                    #show_tensor_distribution(data_tensor)

                    end_time = time.time()
                    times.append(end_time - start_time)
                    print(f"本epoch運行 {end_time - start_time} 秒")

        else:

            # train
            model.train()
            
            with torch.no_grad():
                data_tensor = torch.zeros(target_size, dtype=torch.float32)#.to(device)
                weight_tensor = torch.zeros(target_size, dtype=torch.float32, requires_grad=False)#.to(device)
                

            for epoch in range(epochs):
                start_time = time.time()

                loss_sum = 0
                #with torch.no_grad():
                #    data_tensor = torch.zeros(target_size).to(device)
                #    weight_tensor = torch.zeros(target_size, device=device, dtype=torch.float32, requires_grad=False)
                data_tensor.zero_()
                weight_tensor.zero_()
                
                for i in range(len(index_x)):
                    for j in range(len(index_y)):
                        loss_value = 0
                        print('epoch:', epoch, 'i:', i, 'j:', j)
                        #optimizer.param_groups[0]['lr'] = optimizer.param_groups[0]['lr'] * learning_rate_decay
                        #print('learning rate = ', optimizer.param_groups[0]['lr'])
                        for k in range(len(index_z)):
                            #optimizer.param_groups[0]['lr'] = optimizer.param_groups[0]['lr'] * learning_rate_decay
                            #print('learning rate = ', optimizer.param_groups[0]['lr'])

                            data = data_sinogram[index_x[i]:index_x[i]+patch_size_[0], index_y[j]:index_y[j]+patch_size_[1], index_z[k]:index_z[k]+patch_size_[2]]
                            data = data.unsqueeze(0).unsqueeze(0)#.to(device)
                            target = target_sinogram[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]]
                            target = target.unsqueeze(0).unsqueeze(0)#.to(device)

                            output = model(data).to(torch.device("cpu"))

                            target = target.squeeze(0).squeeze(0)#.to(device)
                            output = output.squeeze(0).squeeze(0).squeeze(0)#.to(device)

                            #output_temp = output.detach().cpu().numpy()
                            #output_temp = copy.deepcopy(output_temp)
                            #output_temp = torch.tensor(output_temp).to(device)
                            #output_temp = copy.deepcopy(output_temp)
                            
                            output_temp = output.clone().detach()  # 只需要 clone 一次
                            
                            #data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]].detach().cpu().numpy()
                            #data_tensor_temp = copy.deepcopy(data_tensor_temp)
                            #data_tensor_temp = torch.tensor(data_tensor_temp).to(device)
                            
                            data_tensor_temp = data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], 
                               index_y_output[j]:index_y_output[j]+output_patch_size[1], 
                               index_z_output[k]:index_z_output[k]+output_patch_size[2]].clone().detach()

                            output += data_tensor_temp

                            data_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( output_temp )
                            weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]] += ( tensor_ones )

                            output.div_(weight_tensor[index_x_output[i]:index_x_output[i]+output_patch_size[0], index_y_output[j]:index_y_output[j]+output_patch_size[1], index_z_output[k]:index_z_output[k]+output_patch_size[2]])
                            
                            loss = loss_fn(output, target.to(torch.device("cpu")))#.to(device)
                            optimizer.zero_grad()
                            loss.backward() 
                            optimizer.step()
                            
                            loss_value += loss.detach().cpu().item()

                        print('loss:', loss_value)
                        loss_sum += loss_value
                        print('loss_sum:', loss_sum)              

                print('final loss_sum:', loss_sum)
                losses.append(loss_sum)

                #create_folder_if_not_exist(data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram_output' % (target_size[1], target_size[2]))

                #save_sino = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram_output' % (target_size[1], target_size[2])
                #if not os.path.exists(save_sino + '/' + str(loss_sum)):
                    #os.mkdir(save_sino + '/' + str(loss_sum))
                    #pass
                    
                with torch.no_grad():
                    data_tensor /= weight_tensor.clone().detach()
                    
                '''
                for i in range(n):

                    if epoch % 10 == 0:
                        path = save_sino + '/' + str(loss_sum) + '/tomo_' + str("{:05d}".format(rand_start + i)) + '.tif'
                        #tifffile.imsave(path, np.array(data_tensor[i].cpu().detach().numpy()))

                    # save in the next size folder
                    path = data_locate + data_prefix_path + '/proj128_multi_scale/data/trilinear%d_%d_sinogram/' % (target_size[1], target_size[2]) + '/tomo_' + str("{:05d}".format(rand_start + i)) + '.tif'
                    #tifffile.imsave(path, np.array(data_tensor[i].cpu().detach().numpy()))          
                '''
                print(data_tensor.shape)
                #show_tensor_distribution(data_tensor)
                
                #show_slices(data_tensor.cpu().numpy()) # .clone().detach()

                end_time = time.time()
                times.append(end_time - start_time)
                print(f"本epoch運行 {end_time - start_time} 秒")

                if is_save_model:
                    torch.save(model.state_dict(), save_model)
                    torch.save({
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict()
                    }, save_model)
                
                #del weight_tensor
                #del data_sinogram
                #del target_sinogram
                gc.collect()  # 强制进行垃圾回收
                torch.cuda.empty_cache()
                
        if not test_mode:
            eps = list(range(1, len(losses) + 1))
            total_time = sum(times)

            plt.plot(eps, losses, label='Training Loss')
            plt.title('Training Loss Over Epochs')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            #plt.savefig(save_sino + '/Training_Loss.png')
            plt.show()

            plt.plot(eps, times, label='Training Time')
            plt.title('Training Time Over Epochs')
            plt.xlabel('Epoch')
            plt.ylabel('Time')
            plt.legend()
            #plt.savefig(save_sino + '/Training_Time.png')
            plt.show()

            total_params = sum(p.numel() for p in model.parameters())
            '''
            log_path = save_sino + '/log.txt'
            with open(log_path, "w") as file:
                if scale == 'multi_scale' and mode == 'proj_domain':
                    file.write('single scale sinogram training\n')
                    file.write('sinogram size: ' + str(data_size) + ' to ' + str(target_size_list[0]) + '\n')
                if scale == 'multi_scale' and mode == 'image_domain':
                    file.write('single scale recon training\n')
                    file.write('recon size: ' + str(target_size_list[0]) + ' to ' + str(target_size_list[0]) + '\n')

                file.write('patch_size: ' + str(patch_size_) + '\n')
                file.write('overlap_length: ' + str(overlap_length) + '\n')
                file.write('epoch: ' + str(len(losses)) + '\n')
                file.write('loss: ' + str(losses) +  '\n')
                file.write('time: ' +  str(total_time) +  '\n')
                file.write('parameters: ' + str(total_params) +  '\n')
            '''
            if scale == 'multi_scale' and mode == 'proj_domain':
                    print('single scale sinogram training')
                    print('sinogram size: ', str(data_size), ' to ', str(target_size))
            if scale == 'multi_scale' and mode == 'image_domain':
                    print('single scale recon training')
                    print('recon size: ', str(data_size), ' to ', str(target_size))   
            print('patch_size: ', str(patch_size_))
            print('overlap_length: ', str(overlap_length))
            print('epoch: ', str(len(losses)))
            print('loss: ', str(losses))
            print('time: ', str(total_time))
            print('parameters: ', str(total_params))
        
        #training_tensor = data_tensor
        #first_time = False
        if target_size[1] != 800 and mode == 'proj_domain':
            np.save(f'/media/jamescheng/My Passport/LDCT/result/proj{target_size[1]}_output/{data_id}_sinogram.npy', data_tensor.cpu().numpy())
        elif mode == 'proj_domain':
            np.save(f'/media/jamescheng/My Passport/LDCT/result/{data_id}_msnet_sinogram_result2.npy', data_tensor.cpu().numpy())
        elif mode == 'image_domain':
            np.save(f'/media/jamescheng/My Passport/LDCT/result/image_msnet/result/{data_id}_msnet_image_result.npy', data_tensor.cpu().numpy())
        
        del data_tensor
        del weight_tensor
        del data_sinogram
        #del target_sinogram
        gc.collect()  # 强制进行垃圾回收
        torch.cuda.empty_cache()
        
    #res = training_tensor.cpu().numpy()
    #res = interpolate_process(res, 800, 1201)
    #np.save(f'/media/jamescheng/My Passport/LDCT/result/{data_id}_msnet_sinogram_result.npy', training_tensor.cpu().numpy())
    
# multi scale proj-domain train test

# multi scale前段的size

In [ ]:
# recon the proj
import os
import numpy as np
from cone import ConeRec, normalize, showImageTable, evaluateVolume # import the lite version
import time
from scipy import interpolate
from scipy.ndimage import gaussian_filter1d, gaussian_filter
from skimage.transform import rescale, resize

det_width = 1.0
source_origin = 640
origin_det = 384
n_angles = 800
dectector_enlarge = 4

# Data loading
vol_id = 'C002'#'L004_proj.npy', 'L006_proj.npy', 'L012_proj.npy', 'L014_proj.npy', 'L019_proj.npy'
model_name = 'hdnet+'
vol_path = f'/media/jamescheng/My Passport/LDCT/volume/{vol_id}.npy'
proj_path = f'/media/jamescheng/My Passport/LDCT/result/{vol_id}_proj.npy_{model_name}_sinogram_result.npy'
out_dir = f'/media/jamescheng/My Passport/LDCT/result/'

if not os.path.exists(out_dir):
    os.makedirs(out_dir)

vol = np.load(vol_path).astype(np.float32)
proj = np.load(proj_path).astype(np.float32)
vol_shape = vol.shape
vol_slices, vol_rows, vol_columns = vol_shape
vol_max_edge = max(vol_shape)
print('vol', vol_shape, vol_max_edge)
vol = normalize(vol)

# Projection parameters
scan_range = (0, 2 * np.pi)
angles = np.linspace(scan_range[0], scan_range[1], num = n_angles, endpoint = False)
detector_columns = int(vol_max_edge * dectector_enlarge)
detector_rows = int(vol_slices * dectector_enlarge)
proj_shape = (detector_rows, n_angles, detector_columns)
print('proj_shape(detector_rows, angles, detector_columns):', proj_shape)

ct = ConeRec(vol_shape, proj_shape, scan_range = scan_range, angles = angles, volume = vol,
             det_width = det_width, source_origin = source_origin, origin_det =origin_det)
time_s = time.time()
#proj = ct.project()
print('projection time:', time.time() - time_s)
print('proj', proj.shape, proj.dtype)
#imgset = [normalize(proj[:, 0, :]),
#          normalize(proj[:, n_angles//4, :]),
#          normalize(proj[:, n_angles//2, :]),
#          normalize(proj[:, 3*n_angles//4, :]),
#          normalize(proj[:, -1, :]),]
#showImageTable(imgset, 1, len(imgset), figsize=(10, 5))

# reconstruction
time_s = time.time()
rec = ct.reconstruct(proj)
rec = np.clip(rec, 0.0, None)
np.save(f'/media/jamescheng/My Passport/LDCT/result/image_{model_name}/{vol_id}.npy', rec)
#rec = normalize(rec)

for i in range(0, 1):
    #crop_3d_array_inplace(rec, i)
    #crop_3d_array_inplace(vol, i)
    #print('rec', rec.shape)
    #print('reconstruction time:', time.time() - time_s)
    print(i)
    print('MAE, MSE, SSIM, PSNR:', evaluateVolume(rec, vol))

    imgset = [vol[0],
              vol[vol_slices//4],
              vol[vol_slices//2],
              vol[vol_slices//4 * 3],
              vol[-1],]
    showImageTable(imgset, 1, len(imgset), figsize=(10, 5))
    imgset = [rec[0],
              rec[vol_slices//4],
              rec[vol_slices//2],
              rec[vol_slices//4 * 3],
              rec[-1],]
    showImageTable(imgset, 1, len(imgset), figsize=(10, 5))

# outputing
n_vx = proj.shape[0] * proj.shape[1] * proj.shape[2]
data_size = n_vx * 4
print('n_vx', n_vx)
print('data_size, %dbyte, %0.1fGB'%(data_size, (data_size / 2**30)))
time_s = time.time()

#out_path = out_dir + vol_id + '_proj.npz'
#np.savez_compressed(out_path, proj = proj)
#out_path = out_dir + vol_id + '_proj.npy'
#np.save(out_path, proj)
#print('proj save time:', time.time() - time_s)

In [ ]:
# eval the image domain
import os
import numpy as np
from cone import ConeRec, normalize, showImageTable, evaluateVolume # import the lite version
import time
from scipy import interpolate
from scipy.ndimage import gaussian_filter1d, gaussian_filter
from skimage.transform import rescale, resize

vol_id_list = ['N005', 'N012', 'N300', 'C002', 'C004', 'C012', 'L004', 'L006', 'L014', 'L019']

for vol_id in vol_id_list:
    vol_path = f'/media/jamescheng/My Passport/LDCT/volume/{vol_id}.npy'
    rec_path = f'/media/jamescheng/My Passport/LDCT/result/image_msnet/result/{vol_id}.npy_msnet_image_result.npy'

    rec = np.load(rec_path)
    vol = np.load(vol_path)

    print(vol_id, end="") 
    evaluateVolume(rec, vol)

In [ ]:
# crop
def crop_3d_array_inplace(arr, n):
    if n <= 0:
        return
    
    D, H, W = arr.shape
    #if 2 * n >= D or 2 * n >= H or 2 * n >= W:
        #raise ValueError("size erroe")

    arr[:n, :, :] = 0  
    arr[-n:, :, :] = 0  
    arr[:, :n, :] = 0  
    arr[:, -n:, :] = 0 
    arr[:, :, :n] = 0  
    arr[:, :, -n:] = 0  

arr = np.random.rand(4, 4, 4)
print(arr)
crop_3d_array_inplace(arr, 1)
print(arr)

In [ ]:
# create proj 100, 200, 400
import torch
import torch.nn.functional as F
import numpy as np

def interpolate_torch(im, before_view = 1201, after_view = 50):
    # Step 1: Convert the NumPy array to a Torch tensor
    im_torch = torch.tensor(im, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # Add batch and channel dimensions

    # Step 2: Define the target size for interpolation
    # Assuming downsampling the second dimension to 150
    target_size = (im_torch.shape[2], after_view, im_torch.shape[4])

    # Step 3: Perform interpolation using Torch
    im_interpolated = F.interpolate(im_torch, size=target_size, mode='trilinear', align_corners=False)

    # Step 4: Remove the added dimensions to return to original shape
    im_interpolated = im_interpolated.squeeze(0).squeeze(0)

    # Convert back to NumPy array if needed
    im_interpolated_np = im_interpolated.numpy()
    
    return im_interpolated_np

data_locate = '/media/jamescheng/My Passport/LDCT/proj_nml_vol/'
data_list = ['L004_proj.npy']
size_list = [100, 200, 400]
full_view = 800

for data_id in data_list:
    
    for size in size_list:
        
        data_path = f'{data_locate}{data_id}'
        
        output_path = f'/media/jamescheng/My Passport/LDCT/result/proj{size}/{data_id}_sinogram.npy'
        
        projection_gt = np.load(data_path)

        projection_target = interpolate_torch(projection_gt, full_view, size)
        
        np.save(output_path, projection_target)

In [ ]:
# N005 100proj to 800proj
import torch
import torch.nn.functional as F
import numpy as np

def interpolate_torch(im, before_view = 1201, after_view = 50):
    # Step 1: Convert the NumPy array to a Torch tensor
    im_torch = torch.tensor(im, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # Add batch and channel dimensions

    # Step 2: Define the target size for interpolation
    # Assuming downsampling the second dimension to 150
    target_size = (im_torch.shape[2], after_view, im_torch.shape[4])

    # Step 3: Perform interpolation using Torch
    im_interpolated = F.interpolate(im_torch, size=target_size, mode='trilinear', align_corners=False)

    # Step 4: Remove the added dimensions to return to original shape
    im_interpolated = im_interpolated.squeeze(0).squeeze(0)

    # Convert back to NumPy array if needed
    im_interpolated_np = im_interpolated.numpy()
    
    return im_interpolated_np

data_locate = '/home/jamescheng/Documents/leo_paper_img/fdk_result/'
data_id = 'N005'

data_path = f'{data_locate}{data_id}_proj100.npy'

output_path = f'{data_locate}{data_id}_100proj_to_800proj.npy'

projection_gt = np.load(data_path)

projection_target = interpolate_torch(projection_gt, 100, 800)

np.save(output_path, projection_target)

In [ ]:
# recon the N005_100proj and N005_100proj_to_800proj
import os
import numpy as np
from cone import ConeRec, normalize, showImageTable, evaluateVolume # import the lite version
import time
from scipy import interpolate
from scipy.ndimage import gaussian_filter1d, gaussian_filter
from skimage.transform import rescale, resize

det_width = 1.0
source_origin = 640
origin_det = 384
n_angles = 800
dectector_enlarge = 4

# Data loading
vol_id = 'N005_800proj'
data_locate = '/home/jamescheng/Documents/leo_paper_img/multi_ablation/'
vol_path = f'/media/jamescheng/My Passport/LDCT/volume/{vol_id[0:4]}.npy'
proj_path = f'{data_locate}{vol_id}.npy'

vol = np.load(vol_path).astype(np.float32)
proj = np.load(proj_path).astype(np.float32)
vol_shape = vol.shape
vol_slices, vol_rows, vol_columns = vol_shape
vol_max_edge = max(vol_shape)
print('vol', vol_shape, vol_max_edge)
vol = normalize(vol)

# Projection parameters
scan_range = (0, 2 * np.pi)
angles = np.linspace(scan_range[0], scan_range[1], num = n_angles, endpoint = False)
detector_columns = int(vol_max_edge * dectector_enlarge)
detector_rows = int(vol_slices * dectector_enlarge)
proj_shape = (detector_rows, n_angles, detector_columns)
print('proj_shape(detector_rows, angles, detector_columns):', proj_shape)

ct = ConeRec(vol_shape, proj_shape, scan_range = scan_range, angles = angles, volume = vol,
             det_width = det_width, source_origin = source_origin, origin_det =origin_det)
time_s = time.time()
print('projection time:', time.time() - time_s)
print('proj', proj.shape, proj.dtype)

# reconstruction
time_s = time.time()
rec = ct.reconstruct(proj)
rec = np.clip(rec, 0.0, None)
np.save(f'{data_locate}/{vol_id}_recon.npy', rec)

for i in range(0, 1):
    print(i)
    print('MAE, MSE, SSIM, PSNR:', evaluateVolume(rec, vol))

    imgset = [vol[0],
              vol[vol_slices//4],
              vol[vol_slices//2],
              vol[vol_slices//4 * 3],
              vol[-1],]
    showImageTable(imgset, 1, len(imgset), figsize=(10, 5))
    imgset = [rec[0],
              rec[vol_slices//4],
              rec[vol_slices//2],
              rec[vol_slices//4 * 3],
              rec[-1],]
    showImageTable(imgset, 1, len(imgset), figsize=(10, 5))

# outputing
n_vx = proj.shape[0] * proj.shape[1] * proj.shape[2]
data_size = n_vx * 4
print('n_vx', n_vx)
print('data_size, %dbyte, %0.1fGB'%(data_size, (data_size / 2**30)))
time_s = time.time()

In [ ]:
# read image
import numpy as np
import matplotlib.pyplot as plt

vol_id = 'N005'
model = 'hdnet+'

#/home/jamescheng/Documents/leo_paper_img/volume_result
#img = np.load(f'/media/jamescheng/My Passport/LDCT/result/image_{model}/result/{vol_id}.npy_{model}_image.npy')
img = np.load(f'/home/jamescheng/Documents/leo_paper_img/multi_ablation/N005_800proj_recon.npy')
vol = np.load(f'/media/jamescheng/My Passport/LDCT/volume/{vol_id}.npy')
vol_slices = img.shape[0]

print('rec: ', img.shape)
print('vol: ', vol.shape)

print('rec[0]')
plt.imshow(img[0], cmap='gray')
plt.axis('off')
plt.show()
print('vol[0]')
plt.imshow(vol[0], cmap='gray')
plt.axis('off')
plt.show()

print('rec[vol_slices//4]')
plt.imshow(img[vol_slices//4], cmap='gray')
plt.axis('off')
plt.show()
print('vol[vol_slices//4]')
plt.imshow(vol[vol_slices//4], cmap='gray')
plt.axis('off')
plt.show()

print('rec[vol_slices*2//4]')
plt.imshow(img[vol_slices*2//4], cmap='gray')
plt.axis('off')
plt.show()
print('vol[vol_slices*2//4]')
plt.imshow(vol[vol_slices*2//4], cmap='gray')
plt.axis('off')
plt.show()

print('rec[vol_slices*3//4]')
plt.imshow(img[vol_slices*3//4], cmap='gray')
plt.axis('off')
plt.show()
print('vol[vol_slices*3//4]')
plt.imshow(vol[vol_slices*3//4], cmap='gray')
plt.axis('off')
plt.show()

print('rec[-1]')
plt.imshow(img[-1], cmap='gray')
plt.axis('off')
plt.show()
print('vol[-1]')
plt.imshow(vol[-1], cmap='gray')
plt.axis('off')
plt.show()